# Evaluación Final Transversal — Sistema multimodal de imágenes y texto

## Parte 1: Clasificación de prendas de Fashion-MNIST con una red convolucional (PyTorch)

**Institución:** Duoc UC, Escuela de Informática y Telecomunicaciones<br>
**Asignatura:** Deep Learning (DLY0100) · Sección 003D<br>
**Docente:** Guillermo Eugenio Pinto Fuentes<br>
**Integrantes:** Esteban Pacheco Vega, Francisco Figueroa Donoso<br>
**Fecha:** 24-11-2026<br>
**Repositorio:** https://github.com/Fran-informatica/EFT_EPACHECO_FFIGUEROA_DLY0100_003D

## 1. Introducción

### 1.1 Descripción del problema

El proyecto integra dos modalidades de datos, imágenes de prendas y reseñas de texto, para obtener una predicción conjunta. El trabajo se divide en tres notebooks: este entrena la **red convolucional (CNN)** que clasifica las imágenes; el segundo entrena los modelos de texto (RNN simple, LSTM y Transformer) sobre reseñas de IMDB; el tercero fusiona las representaciones de ambos en un clasificador final.

**Fashion-MNIST** contiene 70.000 imágenes en escala de grises de 28 × 28 píxeles, repartidas en 10 clases de prendas: 60.000 de entrenamiento (6.000 por clase) y 10.000 de prueba (1.000 por clase). Las clases están exactamente balanceadas, no hay imágenes duplicadas dentro de cada conjunto ni compartidas entre ellos, y el 50,2 % de los píxeles es fondo negro (valor 0). La intensidad media por clase va de 0,137 (sandalia) a 0,385 (abrigo), en escala [0, 1]. Estas cifras se calculan en la sección 3.

Una CNN es adecuada para este problema por tres razones:

- **Patrones locales:** los filtros convolucionales detectan bordes, texturas y formas (cuellos, mangas, suelas) en ventanas pequeñas de la imagen.
- **Pesos compartidos:** el mismo filtro recorre toda la imagen, así que el modelo necesita muchos menos parámetros que una red densa que conecta cada píxel con cada neurona.
- **Tolerancia a desplazamientos:** el *pooling* resume cada zona, de modo que una prenda levemente movida produce una representación parecida.

Un **perceptrón multicapa (MLP)** trata cada píxel como una variable independiente y pierde la estructura espacial. La sección 9 compara ambos con evidencia propia.

### 1.2 Objetivo del modelo

Entrenar y optimizar una CNN que clasifique las 10 prendas de Fashion-MNIST y que, además, entregue una **representación visual compacta**: el vector de la última capa oculta, que el notebook de fusión combina con la representación del texto.

- Se optimizan el tamaño del lote, la tasa de aprendizaje y el número de épocas, además del preprocesamiento, las funciones de activación y de pérdida, la arquitectura y la regularización.
- **Métrica de decisión: F1 macro en validación.** Con clases balanceadas coincide casi con la exactitud, pero pondera igual a cada clase y es la misma métrica que usan los notebooks de texto y de fusión, de modo que los tres se comparan con el mismo criterio.

**Nota de terminología:** *accuracy* = **exactitud** (proporción de aciertos); *precision* = **precisión** (de lo que el modelo asigna a una clase, cuánto es correcto); *recall* = **exhaustividad** (de los ejemplos reales de una clase, cuántos detecta); F1 = media armónica de precisión y exhaustividad. El promedio **macro** pondera igual a todas las clases y el **ponderado** según su cantidad de ejemplos.

### 1.3 Estructura del notebook

1. Introducción.
2. Configuración inicial, reutilización de resultados y herramientas.
3. Carga y exploración de los datos, partición y líneas base sin aprendizaje.
4. Preprocesamiento: normalización, aumento de datos y lotes en GPU.
5. Utilidades de entrenamiento, evaluación y experimentos.
6. Arquitectura de la CNN (y del MLP de comparación) y configuración base.
7. Modelo base.
8. Rango de error entre reentrenamientos.
9. Experimento clave: CNN frente a MLP.
10. a 14. Experimentos de preprocesamiento, funciones, entrenamiento, arquitectura y regularización.
15. Modelo final.
16. Evaluación en test.
17. Comparación entre la configuración base y la final.
18. Síntesis de los experimentos.
19. Verificaciones de la dependencia del orden.
20. Representación visual para la fusión.
21. a 23. Conclusiones, conclusión final y referencias.
24. a 27. Anexos (gráficos para la presentación y herramientas), respaldo y descripción de las salidas.

**Criterio metodológico:**

- Todas las decisiones se toman con el conjunto de **validación** (10 % del entrenamiento, estratificado). El conjunto de **test** se usa una sola vez, con el modelo final.
- Se cambia **un hiperparámetro a la vez** sobre la configuración vigente; la variante elegida se incorpora y el experimento siguiente parte de ella.
- Antes de comparar se mide el **rango de error** entre reentrenamientos con 5 semillas. Una diferencia menor o igual a ese rango es un **empate técnico**, y los empates se deciden según el contexto del problema, no solo por la métrica.
- Si la mejor variante queda en el borde del rango probado, el rango se amplía un paso en esa dirección.
- Como la selección de un factor a la vez depende del orden, al final se verifican las combinaciones con interacción conocida y se compara el modelo final con el base usando las mismas semillas.

## 2. Configuración inicial

Se importan en un único bloque todas las librerías del notebook:

- **PyTorch** (`torch.nn`, `torch.nn.functional`): define la CNN, el ciclo de entrenamiento y el aumento de datos en GPU.
- **NumPy / Pandas**: arreglos y tablas comparativas.
- **scikit-learn**: partición estratificada, métricas y análisis de componentes principales (PCA) de la representación.
- **Plotly**: gráficos interactivos. El renderizador `vscode+colab` permite verlos tanto en Colab como en VS Code conectado a un servidor de Colab.
- **IPython.display**: tablas con `display`.
- **gzip / hashlib / urllib / os / json / shutil**: lectura de los archivos (comprimidos o no), detección de duplicados, descarga desde el repositorio y respaldo.
- **copy / random / time**: copia de los pesos de la mejor época, semillas y tiempos.

`fijar_semilla(semilla)` fija las semillas de `random`, NumPy y PyTorch (CPU y GPU). No retorna nada.

In [ ]:
# ============================================================
# Importación de librerías (todos los imports del notebook van aquí)
# ============================================================
import copy
import gzip
import hashlib
import json
import os
import random
import shutil
import time
import urllib.error
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# ============================================================
# Reproducibilidad, dispositivo y visualización
# ============================================================
SEED = 42


def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


fijar_semilla(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pio.renderers.default = "vscode+colab"
COLORES = ["#104281", "#3987E5", "#86B6EF"]
AMARILLO = "#F7B500"
print("PyTorch", torch.__version__, "| dispositivo:", device)

**Nota:** aun con semilla fija, el entrenamiento en GPU no es completamente determinista. Por eso, la sección 8 mide ese margen de variación antes de comparar configuraciones.

**Requisito de ejecución:** activar GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*).

### 2.1 Reutilización de resultados ya obtenidos

Para no repetir cómputo, las métricas de cada corrida se guardan en `resultados_cnn_guardados.json` y se reutilizan cuando la configuración y la semilla coinciden **exactamente**: la clave de cada corrida incluye todos los hiperparámetros, la semilla, el tope de épocas y la paciencia.

- Las corridas que necesitan el modelo entrenado o sus curvas (modelo base, activación, pérdida, épocas y modelo final) se entrenan siempre.
- Cada corrida nueva se imprime en la salida de su celda como una línea `CORRIDA_GUARDADA {...}`. Si la sesión se corta, el JSON se reconstruye desde el notebook guardado, sin montar Google Drive.
- `leer_archivo(ruta, binario)` busca el archivo en la carpeta de trabajo y, si no está, lo descarga de `URL_REPO` (la dirección *raw* del repositorio del grupo). Retorna el contenido, o `None` si no está en ninguno de los dos lugares.
- `guardar_corridas()` escribe en el JSON las corridas guardadas más las nuevas.

In [ ]:
# ============================================================
# Carga de los resultados guardados
# ============================================================
REUTILIZAR_RESULTADOS = True
ARCHIVO_RESULTADOS = "resultados_cnn_guardados.json"
URL_REPO = "https://raw.githubusercontent.com/Fran-informatica/EFT_EPACHECO_FFIGUEROA_DLY0100_003D/main/"


def leer_archivo(ruta, binario=False):
    if os.path.exists(ruta):
        with open(ruta, "rb") as f:
            datos = f.read()
    else:
        try:
            with urllib.request.urlopen(URL_REPO + urllib.parse.quote(ruta)) as r:
                datos = r.read()
        except urllib.error.HTTPError:
            return None
    return datos if binario else datos.decode("utf-8")


def guardar_corridas():
    with open(ARCHIVO_RESULTADOS, "w", encoding="utf-8") as f:
        json.dump({**RESULTADOS_GUARDADOS, **CORRIDAS_NUEVAS}, f, ensure_ascii=False, indent=1, default=float)


texto_resultados = leer_archivo(ARCHIVO_RESULTADOS) if REUTILIZAR_RESULTADOS else None
RESULTADOS_GUARDADOS = json.loads(texto_resultados) if texto_resultados else {}
CORRIDAS_NUEVAS = {}
print(f"Corridas guardadas disponibles: {len(RESULTADOS_GUARDADOS)}")

### 2.2 Herramientas: fortalezas y debilidades

| Herramienta | Fortaleza en este proyecto | Debilidad o costo |
|---|---|---|
| **PyTorch** | Ciclo de entrenamiento explícito: se controla el Early Stopping, el aumento de datos en GPU y la extracción de la capa oculta que necesita la fusión. Es el mismo framework de los modelos de texto, de modo que las tres redes se combinan sin conversiones | Más código que Keras para lo mismo (ciclo de épocas, métricas, guardado); los errores de forma de los tensores aparecen en ejecución |
| **Google Colab con GPU T4** | Entrena una época de Fashion-MNIST en segundos; el equipo local no tiene GPU compatible con CUDA | Cuota de uso limitada y sesiones que se cortan; los archivos quedan en el servidor y hay que descargarlos |
| **Datos completos en la GPU** | Las 70.000 imágenes ocupan 55 MB como enteros de 8 bits; mantenerlas en la GPU evita copiar cada lote desde la CPU | No escala a datasets grandes; con imágenes de mayor resolución habría que volver a un `DataLoader` |
| **Plotly** | Gráficos interactivos (zoom, valores al pasar el cursor) y exportables a HTML y PNG para la presentación | Los gráficos pesan más en el notebook que una imagen estática |
| **scikit-learn** | Métricas estándar y verificadas (F1, precisión, exhaustividad, matriz de confusión) y partición estratificada | Trabaja en CPU con NumPy: las predicciones deben copiarse desde la GPU |

## 3. Carga y exploración de los datos

### 3.1 Lectura de los archivos IDX

Los datos se leen de los cuatro archivos del formato **IDX** (imágenes y etiquetas de entrenamiento y de test), ubicados en la raíz del repositorio. Se usan estos archivos, y no una descarga de una librería, para trabajar exactamente con los datos entregados y no depender de un servidor externo.

`leer_idx(nombre)` lee el archivo (si está comprimido en gzip, lo descomprime) y luego su cabecera: el cuarto byte indica el número de dimensiones y cada dimensión ocupa 4 bytes; el resto son los valores en enteros de 8 bits. Retorna un arreglo de NumPy con la forma indicada en la cabecera.

In [ ]:
# ============================================================
# Lectura de los archivos IDX
# ============================================================
CARPETA_DATOS = ""
ARCHIVOS = {"x_train": "train-images-idx3-ubyte", "y_train": "train-labels-idx1-ubyte",
            "x_test": "t10k-images-idx3-ubyte", "y_test": "t10k-labels-idx1-ubyte"}
CLASES = ["Camiseta / top", "Pantalón", "Suéter", "Vestido", "Abrigo",
          "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]


def leer_idx(nombre):
    ruta = f"{CARPETA_DATOS}/{nombre}" if CARPETA_DATOS else nombre
    datos = leer_archivo(ruta, binario=True)
    if datos is None:
        raise FileNotFoundError(f"No se encontró {ruta} ni en la carpeta de trabajo ni en el repositorio")
    if datos[:2] == b"\x1f\x8b":
        datos = gzip.decompress(datos)
    dimensiones = datos[3]
    forma = [int.from_bytes(datos[4 + 4 * i:8 + 4 * i], "big") for i in range(dimensiones)]
    return np.frombuffer(datos, dtype=np.uint8, offset=4 + 4 * dimensiones).reshape(forma)


x_train_completo = leer_idx(ARCHIVOS["x_train"])
y_train_completo = leer_idx(ARCHIVOS["y_train"]).astype(np.int64)
x_test = leer_idx(ARCHIVOS["x_test"])
y_test = leer_idx(ARCHIVOS["y_test"]).astype(np.int64)
print("Entrenamiento:", x_train_completo.shape, "| Test:", x_test.shape)

### 3.2 Validación de los datos

Antes de entrenar se verifica que los datos sean consistentes: igual número de imágenes y etiquetas, valores de píxel entre 0 y 255, etiquetas entre 0 y 9 y ausencia de imágenes duplicadas. Una imagen repetida dentro del entrenamiento pesa doble, y una imagen de test que también está en entrenamiento infla la métrica final.

`huellas(imagenes)` calcula una huella MD5 por imagen (dos imágenes idénticas byte a byte tienen la misma huella). Retorna una lista de cadenas.

In [ ]:
# ============================================================
# Validación de formas, rangos y duplicados
# ============================================================
def huellas(imagenes):
    return [hashlib.md5(img.tobytes()).hexdigest() for img in imagenes]


huellas_train, huellas_test = huellas(x_train_completo), huellas(x_test)
validacion = pd.DataFrame({
    "comprobación": ["imágenes = etiquetas (train)", "imágenes = etiquetas (test)", "píxel mínimo / máximo",
                     "etiquetas distintas", "duplicados en train", "duplicados en test", "imágenes de test presentes en train"],
    "resultado": [len(x_train_completo) == len(y_train_completo), len(x_test) == len(y_test),
                  f"{x_train_completo.min()} / {x_train_completo.max()}", len(np.unique(y_train_completo)),
                  len(huellas_train) - len(set(huellas_train)), len(huellas_test) - len(set(huellas_test)),
                  len(set(huellas_train) & set(huellas_test))],
})
display(validacion)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Coinciden imágenes y etiquetas y los rangos son los esperados?
- Número de duplicados dentro de cada conjunto y entre ellos (si hay, decidir si se eliminan).

### 3.3 Distribución de clases

Se cuenta cuántas imágenes tiene cada clase en entrenamiento y en test. Si hubiera desbalance, la exactitud podría ser alta solo por acertar las clases frecuentes, y habría que probar una corrección (pesos de clase o sobremuestreo).

In [ ]:
# ============================================================
# Conteo de imágenes por clase
# ============================================================
tabla_clases = pd.DataFrame({
    "clase": CLASES,
    "train": np.bincount(y_train_completo, minlength=10),
    "test": np.bincount(y_test, minlength=10),
})
tabla_clases["% train"] = tabla_clases["train"] / tabla_clases["train"].sum() * 100
display(tabla_clases.round(2))

In [ ]:
# ============================================================
# Gráfico de la distribución de clases
# ============================================================
fig = go.Figure([go.Bar(x=CLASES, y=tabla_clases["train"], name="entrenamiento", marker_color=COLORES[0]),
                 go.Bar(x=CLASES, y=tabla_clases["test"], name="test", marker_color=COLORES[2])])
fig.update_layout(title="Imágenes por clase", barmode="group", yaxis_title="imágenes", height=420)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Proporción de cada clase en entrenamiento y test.
- Si las clases están balanceadas, indicar que no se necesita corrección del desbalance y por qué.

### 3.4 Ejemplos por clase

Se muestran dos imágenes de cada clase, elegidas al azar con la semilla fija, para reconocer qué clases se parecen visualmente (por ejemplo, camiseta, camisa, suéter y abrigo) y anticipar dónde se concentrarán los errores.

In [ ]:
# ============================================================
# Dos ejemplos por clase
# ============================================================
generador_np = np.random.default_rng(SEED)
indices_ejemplo = [i for c in range(10) for i in generador_np.choice(np.where(y_train_completo == c)[0], 2, replace=False)]
fig = px.imshow(x_train_completo[indices_ejemplo], facet_col=0, facet_col_wrap=10, color_continuous_scale="gray_r",
                facet_row_spacing=0.08, height=330)
fig.for_each_annotation(lambda a: a.update(text=CLASES[y_train_completo[indices_ejemplo[int(a.text.split("=")[1])]]]))
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(title="Ejemplos por clase", coloraxis_showscale=False)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Clases visualmente parecidas que podrían confundirse.
- Características que distinguen a las clases (siluetas, mangas, suelas, asas).

### 3.5 Intensidad de los píxeles

La intensidad media y la proporción de píxeles encendidos (mayores que 0) por clase indican cuánto ocupa la prenda en la imagen. Además, la distribución de los valores de píxel justifica la normalización de la sección 4: con valores entre 0 y 255 las activaciones iniciales son grandes y el entrenamiento se vuelve inestable.

In [ ]:
# ============================================================
# Intensidad media y píxeles encendidos por clase
# ============================================================
tabla_intensidad = pd.DataFrame({
    "clase": CLASES,
    "intensidad_media": [x_train_completo[y_train_completo == c].mean() / 255 for c in range(10)],
    "pixeles_encendidos": [(x_train_completo[y_train_completo == c] > 0).mean() for c in range(10)],
})
print(f"Píxeles en cero en entrenamiento: {(x_train_completo == 0).mean() * 100:.1f} %")
display(tabla_intensidad.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Clases con más y menos intensidad o superficie ocupada.
- Proporción de fondo (píxeles en cero) y su relación con el desplazamiento del aumento de datos.

### 3.6 Partición en entrenamiento y validación

El conjunto de test (10.000 imágenes) se reserva para la evaluación final. Del entrenamiento se separa un **10 % estratificado** para validación (6.000 imágenes, 600 por clase), con la semilla fija: así cada clase mantiene su proporción y la partición es reproducible. Los índices se guardan en el respaldo para que el notebook de fusión use la misma partición.

In [ ]:
# ============================================================
# Partición estratificada 90 / 10
# ============================================================
idx_train, idx_val = train_test_split(np.arange(len(y_train_completo)), test_size=0.1,
                                      stratify=y_train_completo, random_state=SEED)
x_train, y_train = x_train_completo[idx_train], y_train_completo[idx_train]
x_val, y_val = x_train_completo[idx_val], y_train_completo[idx_val]
print(f"Entrenamiento: {len(y_train)} | Validación: {len(y_val)} | Test: {len(y_test)}")
print("Imágenes por clase en validación:", np.bincount(y_val, minlength=10))

### 3.7 Líneas base sin aprendizaje

Las líneas base fijan el piso que la CNN debe superar:

- **Clase mayoritaria:** predice siempre la clase más frecuente del entrenamiento. Con clases balanceadas, su exactitud es de un 10 %.
- **Prototipo medio por clase:** calcula la imagen promedio de cada clase y asigna cada imagen a la clase con el promedio más cercano (distancia euclidiana). No usa gradientes ni pesos: mide cuánto se resuelve solo con la silueta promedio.

`metricas(y_true, y_pred)` retorna un diccionario con exactitud, precisión macro, exhaustividad macro, F1 macro y F1 ponderado. `predecir_prototipo(x, prototipos)` retorna la clase del prototipo más cercano a cada imagen.

In [ ]:
# ============================================================
# Métricas y líneas base sin aprendizaje
# ============================================================
def metricas(y_true, y_pred):
    precision, exhaustividad, _, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"exactitud": accuracy_score(y_true, y_pred), "precision_macro": precision,
            "exhaustividad_macro": exhaustividad, "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "f1_ponderado": f1_score(y_true, y_pred, average="weighted")}


def predecir_prototipo(x, prototipos):
    planas = x.reshape(len(x), -1).astype(np.float64) / 255
    distancias = (planas ** 2).sum(1, keepdims=True) - 2 * planas @ prototipos.T + (prototipos ** 2).sum(1)
    return distancias.argmin(1)


CLASE_MAYORITARIA = int(np.bincount(y_train).argmax())
PROTOTIPOS = np.stack([x_train[y_train == c].reshape(-1, 784).mean(0) / 255 for c in range(10)])
tabla_lineas_base_val = pd.DataFrame([
    {"modelo": "Clase mayoritaria", **metricas(y_val, np.full(len(y_val), CLASE_MAYORITARIA))},
    {"modelo": "Prototipo medio por clase", **metricas(y_val, predecir_prototipo(x_val, PROTOTIPOS))},
])
display(tabla_lineas_base_val.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de cada línea base en validación: es el piso de la CNN.
- Qué clases resuelve bien el prototipo medio y cuáles no (si se revisa por clase).

## 4. Preprocesamiento

### 4.1 Normalización de los píxeles

Se definen tres formas de escalar los píxeles, que la sección 10 compara:

- **Sin escalar (0–255):** los valores originales. Producen activaciones y gradientes grandes al inicio del entrenamiento.
- **Escala [0, 1]:** divide por 255. Es la opción más simple y mantiene el fondo en 0.
- **Estandarizada:** resta la media y divide por la desviación estándar de los píxeles de entrenamiento, de modo que la entrada queda centrada en 0 con varianza 1. La media y la desviación se calculan **solo con el entrenamiento** para no filtrar información de validación ni de test.

El gráfico compara la distribución de los valores de una muestra de imágenes con cada escala.

In [ ]:
# ============================================================
# Estadísticos de normalización (solo entrenamiento)
# ============================================================
MEDIA_PIXEL = float(x_train.mean() / 255)
STD_PIXEL = float(x_train.std() / 255)
print(f"Media de los píxeles: {MEDIA_PIXEL:.4f} | Desviación estándar: {STD_PIXEL:.4f}")

In [ ]:
# ============================================================
# Distribución de los valores con cada escala
# ============================================================
muestra = x_train[:500].reshape(-1).astype(np.float64)
escalas = {"Sin escalar (0–255)": muestra, "Escala [0, 1]": muestra / 255,
           "Estandarizada": (muestra / 255 - MEDIA_PIXEL) / STD_PIXEL}
fig = make_subplots(rows=1, cols=3, subplot_titles=list(escalas))
for col, valores in enumerate(escalas.values(), start=1):
    fig.add_trace(go.Histogram(x=valores, nbinsx=50, marker_color=COLORES[0], showlegend=False), row=1, col=col)
fig.update_yaxes(type="log", title_text="frecuencia (escala log)", row=1, col=1)
fig.update_layout(title="Valores de los píxeles según la normalización", height=380)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Rango de valores de cada escala y lugar donde queda el fondo (valor 0 original).
- Qué escala se espera que entrene mejor y por qué (se comprueba en la sección 10).

### 4.2 Aumento de datos

El aumento de datos genera variantes de las imágenes de entrenamiento en cada época, para que el modelo no memorice posiciones exactas. Se aplican dos transformaciones que no cambian la clase de una prenda:

- **Volteo horizontal** con probabilidad 0,5: una prenda reflejada sigue siendo la misma prenda.
- **Desplazamiento** de hasta 2 píxeles en cada eje: las prendas no siempre están perfectamente centradas.

No se usan rotaciones grandes ni volteos verticales, porque una prenda boca abajo no aparece en los datos reales. El aumento se aplica en la GPU sobre cada lote y solo durante el entrenamiento; la validación y el test usan las imágenes originales.

`aumentar(x, generador, desplazamiento)` recibe un lote de imágenes como tensor, el generador de números aleatorios y el desplazamiento máximo en píxeles. Retorna el lote transformado (los bordes que quedan vacíos se rellenan con fondo negro).

In [ ]:
# ============================================================
# Aumento de datos en GPU
# ============================================================
def aumentar(x, generador, desplazamiento=2):
    n = x.shape[0]
    voltear = torch.rand(n, generator=generador, device=x.device) < 0.5
    x = torch.where(voltear[:, None, None, None], x.flip(3), x)
    # Traslación en coordenadas normalizadas: 1 píxel = 2 / 28
    traslacion = (torch.rand(n, 2, generator=generador, device=x.device) * 2 - 1) * desplazamiento * 2 / 28
    theta = torch.zeros(n, 2, 3, device=x.device)
    theta[:, 0, 0] = 1
    theta[:, 1, 1] = 1
    theta[:, :, 2] = traslacion
    malla = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, malla, padding_mode="zeros", align_corners=False)

In [ ]:
# ============================================================
# Comparación visual: originales frente a aumentadas
# ============================================================
generador_vista = torch.Generator(device=device).manual_seed(SEED)
originales = torch.tensor(x_train[:6], dtype=torch.float32, device=device).unsqueeze(1)
aumentadas = aumentar(originales, generador_vista)
comparacion_aumento = torch.cat([originales, aumentadas]).squeeze(1).cpu().numpy()
fig = px.imshow(comparacion_aumento, facet_col=0, facet_col_wrap=6, color_continuous_scale="gray_r", height=360)
fig.for_each_annotation(lambda a: a.update(text="original" if int(a.text.split("=")[1]) < 6 else "aumentada"))
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(title="Imágenes originales (fila superior) y aumentadas (fila inferior)", coloraxis_showscale=False)
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Las imágenes aumentadas conservan la clase reconocible.
- Si el desplazamiento recorta partes de la prenda (se compara su efecto en la sección 10).

### 4.3 Tensores en GPU y preparación de los lotes

Las imágenes se copian una sola vez a la GPU como enteros de 8 bits, con la forma `[imágenes, 1 canal, 28, 28]` que espera una capa convolucional. Cada lote se convierte a decimales, se aumenta (solo en entrenamiento) y se normaliza en el momento.

`preparar_lote(x, config, generador)` recibe el lote en enteros de 8 bits, la configuración (claves `normalizacion` y `aumento`) y, solo en entrenamiento, el generador aleatorio. Retorna el lote listo para el modelo. El aumento se aplica antes de normalizar, para que el relleno de los bordes quede como fondo negro en cualquier escala.

In [ ]:
# ============================================================
# Tensores en GPU y preparación de cada lote
# ============================================================
X_TRAIN = torch.tensor(x_train, device=device).unsqueeze(1)
Y_TRAIN = torch.tensor(y_train, device=device)
X_VAL = torch.tensor(x_val, device=device).unsqueeze(1)
Y_VAL = torch.tensor(y_val, device=device)
X_TEST = torch.tensor(x_test, device=device).unsqueeze(1)
Y_TEST = torch.tensor(y_test, device=device)


def preparar_lote(x, config, generador=None):
    x = x.float()
    if generador is not None and config["aumento"]:
        x = aumentar(x, generador)
    if config["normalizacion"] == "escala_0_1":
        x = x / 255
    elif config["normalizacion"] == "estandar":
        x = (x / 255 - MEDIA_PIXEL) / STD_PIXEL
    return x


print("Forma del tensor de entrenamiento:", tuple(X_TRAIN.shape), "| tipo:", X_TRAIN.dtype)

## 5. Utilidades de entrenamiento, evaluación y experimentos

### 5.1 Función de pérdida, optimizador, una época y predicción

- `construir_criterio(nombre)`: retorna la función de pérdida de entrenamiento. `entropia_cruzada` es la estándar para clasificación; `suavizado_etiquetas` es la misma con *label smoothing* de 0,1 (la etiqueta correcta vale 0,9 y el resto se reparte entre las demás clases, lo que castiga la sobreconfianza); `mse` es el error cuadrático medio entre las probabilidades y la etiqueta codificada en *one-hot*.
- `construir_optimizador(config, modelo)`: retorna Adam o SGD con *momentum* 0,9, con la tasa de aprendizaje y el *weight decay* (penalización L2) de la configuración.
- `entrenar_epoca(modelo, x, y, config, optimizador, criterio, generador)`: recorre los lotes en un orden aleatorio, calcula la pérdida, hace *backpropagation* y actualiza los pesos. Retorna la pérdida y la exactitud de entrenamiento.
- `predecir(modelo, x, y, config, criterio)`: evalúa sin gradientes en lotes de 2.048 imágenes. Retorna la pérdida (si se entrega un criterio), las etiquetas reales, las predichas y las probabilidades.

In [ ]:
# ============================================================
# Pérdida, optimizador, entrenamiento de una época y predicción
# ============================================================
def construir_criterio(nombre):
    if nombre == "entropia_cruzada":
        return nn.CrossEntropyLoss()
    if nombre == "suavizado_etiquetas":
        return nn.CrossEntropyLoss(label_smoothing=0.1)
    if nombre == "mse":
        return lambda logits, y: F.mse_loss(torch.softmax(logits, 1), F.one_hot(y, 10).float())
    raise ValueError(f"Pérdida desconocida: {nombre}")


def construir_optimizador(config, modelo):
    if config["optimizador"] == "adam":
        return torch.optim.Adam(modelo.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    return torch.optim.SGD(modelo.parameters(), lr=config["learning_rate"], momentum=0.9,
                           weight_decay=config["weight_decay"])


def entrenar_epoca(modelo, x, y, config, optimizador, criterio, generador):
    modelo.train()
    perdida, aciertos, total = 0.0, 0, 0
    orden = torch.randperm(len(y), generator=generador, device=y.device)
    for inicio in range(0, len(y), config["batch_size"]):
        indices = orden[inicio:inicio + config["batch_size"]]
        xb, yb = preparar_lote(x[indices], config, generador), y[indices]
        optimizador.zero_grad()
        logits = modelo(xb)
        loss = criterio(logits, yb)
        loss.backward()
        optimizador.step()
        perdida += loss.item() * len(yb)
        aciertos += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return perdida / total, aciertos / total


def predecir(modelo, x, y, config, criterio=None, lote=2048):
    modelo.eval()
    perdida, salidas = 0.0, []
    with torch.no_grad():
        for inicio in range(0, len(y), lote):
            logits = modelo(preparar_lote(x[inicio:inicio + lote], config))
            if criterio is not None:
                perdida += criterio(logits, y[inicio:inicio + lote]).item() * len(logits)
            salidas.append(logits)
    logits = torch.cat(salidas)
    return {"perdida": perdida / len(y), "y_true": y.cpu().numpy(), "y_pred": logits.argmax(1).cpu().numpy(),
            "probas": torch.softmax(logits, 1).cpu().numpy()}

### 5.2 Entrenamiento con Early Stopping y registro de métricas

`entrenar_y_registrar(config, semilla, epocas, paciencia, verbose, forzar)` construye el modelo desde un diccionario de configuración y lo entrena con **Early Stopping**: monitorea la pérdida de validación y, al terminar, restaura los pesos de la mejor época. Con `paciencia=0` entrena todas las épocas sin detenerse. Retorna `(fila, historia, modelo)`: la fila de métricas en la mejor época, la historia por época y el modelo entrenado.

- **Régimen uniforme:** todas las corridas usan `EPOCAS_EXPERIMENTO = 20` y `PACIENCIA_EXPERIMENTO = 3`, incluido el modelo final. El tope es provisional: el experimento de épocas sin Early Stopping (sección 12) muestra dónde empieza el sobreajuste.
- **Extensión sin reiniciar:** si la mejor época queda en las dos últimas del tope, la corrida **continúa** hasta `EPOCAS_EXTENDIDAS = 30` en lugar de empezar de nuevo, para no comparar como perdedora una variante que todavía mejoraba.
- La pérdida de validación siempre se mide con la entropía cruzada (`CRITERIO_EVAL`), aunque el modelo se entrene con otra pérdida: así las pérdidas de validación de todas las variantes son comparables.
- Si existe un resultado guardado con la misma clave y `forzar=False`, se reutiliza y no se entrena; en ese caso no hay modelo ni historia.
- Cada corrida nueva se guarda en el JSON y se imprime como `CORRIDA_GUARDADA {...}`.
- Si la pérdida deja de ser finita (el entrenamiento **diverge**, por ejemplo con una tasa demasiado alta para la escala de los píxeles), la corrida se detiene, se marca con `diverge = True` y se registra como **Error / Falla** en vez de interrumpir el notebook.

In [ ]:
# ============================================================
# Entrenamiento con Early Stopping, extensión y registro de métricas
# ============================================================
EPOCAS_EXPERIMENTO = 20
PACIENCIA_EXPERIMENTO = 3
EPOCAS_EXTENDIDAS = 30
CRITERIO_EVAL = nn.CrossEntropyLoss()


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


def clave_corrida(config, semilla, epocas, paciencia):
    return json.dumps({**config, "semilla": semilla, "epocas": epocas, "paciencia": paciencia}, sort_keys=True)


def entrenar_y_registrar(config, semilla=SEED, epocas=None, paciencia=None, verbose=False, forzar=False):
    epocas = epocas or EPOCAS_EXPERIMENTO
    paciencia = PACIENCIA_EXPERIMENTO if paciencia is None else paciencia
    clave = clave_corrida(config, semilla, epocas, paciencia)
    if not forzar and clave in RESULTADOS_GUARDADOS:
        return {**RESULTADOS_GUARDADOS[clave], "origen": "guardado"}, None, None

    fijar_semilla(semilla)
    generador = torch.Generator(device=device).manual_seed(semilla)
    modelo = construir_modelo(config)
    criterio = construir_criterio(config["perdida"])
    optimizador = construir_optimizador(config, modelo)
    historia = {"loss_train": [], "exactitud_train": [], "loss_val": [], "exactitud_val": [], "f1_val": []}
    mejor_loss, mejor_estado, mejor_epoca, sin_mejora, diverge = float("inf"), None, 0, 0, False
    tope, epoca, inicio = epocas, 0, time.time()
    while epoca < tope:
        loss_tr, acc_tr = entrenar_epoca(modelo, X_TRAIN, Y_TRAIN, config, optimizador, criterio, generador)
        val = predecir(modelo, X_VAL, Y_VAL, config, CRITERIO_EVAL)
        acc_val = accuracy_score(val["y_true"], val["y_pred"])
        f1_val = f1_score(val["y_true"], val["y_pred"], average="macro")
        for k, valor in zip(historia, [loss_tr, acc_tr, val["perdida"], acc_val, f1_val]):
            historia[k].append(valor)
        if verbose:
            print(f"Época {epoca + 1:02} | loss train {loss_tr:.4f} | loss val {val['perdida']:.4f} "
                  f"| exactitud val {acc_val:.4f} | F1 macro val {f1_val:.4f}")
        if not (np.isfinite(loss_tr) and np.isfinite(val["perdida"])):
            print(f"  Error / Falla: la pérdida dejó de ser finita en la época {epoca + 1} (el entrenamiento divergió).")
            diverge = True
            break
        if val["perdida"] < mejor_loss:
            mejor_loss, mejor_epoca, sin_mejora = val["perdida"], epoca, 0
            mejor_estado = copy.deepcopy(modelo.state_dict())
        else:
            sin_mejora += 1
            if paciencia and sin_mejora >= paciencia:
                break
        epoca += 1
        if epoca == tope and paciencia and mejor_epoca >= tope - 2 and tope < EPOCAS_EXTENDIDAS:
            print(f"  La mejor época ({mejor_epoca + 1}) está en el tope: se continúa hasta {EPOCAS_EXTENDIDAS} épocas.")
            tope = EPOCAS_EXTENDIDAS
    duracion = time.time() - inicio
    if mejor_estado is None:
        mejor_estado = copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(mejor_estado)
    if paciencia and not diverge and mejor_epoca >= tope - 2:
        print(f"  Aviso: la mejor época ({mejor_epoca + 1}) sigue al final del tope extendido: la variante no convergió.")

    m = mejor_epoca
    fila = {
        "exactitud_train": historia["exactitud_train"][m],
        "exactitud_val": historia["exactitud_val"][m],
        "f1_macro_val": historia["f1_val"][m],
        "loss_val": historia["loss_val"][m],
        "brecha_train_val": historia["exactitud_train"][m] - historia["exactitud_val"][m],
        "mejor_epoca": m + 1,
        "epocas_entrenadas": len(historia["loss_train"]),
        "parametros": contar_parametros(modelo),
        "segundos": duracion,
        "segundos_por_epoca": duracion / len(historia["loss_train"]),
        "diverge": diverge,
    }
    CORRIDAS_NUEVAS[clave] = fila
    print("CORRIDA_GUARDADA " + json.dumps({"clave": clave, "fila": fila}, default=float))
    guardar_corridas()
    return {**fila, "origen": "entrenado"}, historia, modelo

### 5.3 Experimentos controlados, selección y ampliación del rango

- `ejecutar_experimento(nombre, variantes, config_actual, forzar)` entrena (o recupera) las **variantes** `{etiqueta: cambios}`, que modifican un solo hiperparámetro respecto de la configuración actual. Retorna una tabla ordenada por **F1 macro de validación**, con la diferencia respecto de la mejor (`diferencia_vs_mejor`) y si cae dentro del rango de error (`dentro_del_margen`). Las historias de las variantes entrenadas quedan en `historias_ultimo_experimento`. Si se repite un experimento, la configuración vuelve a su estado previo y se descartan las decisiones posteriores.
- `ampliar_rango(nombre, variantes, config_actual, hiperparametro, arriba, abajo, minimo, maximo)`: si la mejor variante queda en el **borde** del rango probado, agrega el valor siguiente en esa dirección (funciones `arriba` y `abajo`) y lo entrena, mientras el ganador siga en el borde y no se supere el límite. Retorna la tabla actualizada.
- `seleccionar_variante(tabla)`: **propone** la mejor variante y lista las que están en empate técnico. No decide los empates: esos se revisan caso a caso y la decisión queda en una línea `# Cambio Manual`.
- `aplicar_variante(nombre, variantes, elegida)`: incorpora la variante a `CONFIG_ACTUAL`, registra la decisión con su F1 y el de la configuración previa, y muestra la configuración.
- `tabla_de(nombre)`: retorna la tabla guardada de un experimento.

In [ ]:
# ============================================================
# Experimentos controlados (un hiperparámetro a la vez)
# ============================================================
resultados_experimentos = []
historial_experimentos = []
decisiones = {}
historias_ultimo_experimento = {}
COLUMNAS_TABLA = ["variante", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca",
                  "epocas_entrenadas", "parametros", "segundos_por_epoca", "diferencia_vs_mejor", "dentro_del_margen",
                  "diverge", "origen"]


def entrenar_variantes(nombre, variantes, config_actual, forzar=False):
    filas = []
    for etiqueta, cambios in variantes.items():
        fila, historia, _ = entrenar_y_registrar({**config_actual, **cambios}, forzar=forzar)
        print(f"[{nombre}] {etiqueta}: {fila['origen']}")
        if historia is not None:
            historias_ultimo_experimento[etiqueta] = historia
        filas.append({"experimento": nombre, "variante": etiqueta, **fila})
    return filas


def armar_tabla(filas):
    tabla = pd.DataFrame(filas)
    tabla["diferencia_vs_mejor"] = tabla["f1_macro_val"].max() - tabla["f1_macro_val"]
    tabla["dentro_del_margen"] = tabla["diferencia_vs_mejor"] <= MARGEN_ERROR
    return tabla.sort_values("f1_macro_val", ascending=False).reset_index(drop=True)


def ejecutar_experimento(nombre, variantes, config_actual, forzar=False):
    nombres = [n for n, _ in historial_experimentos]
    if nombre in nombres:
        i = nombres.index(nombre)
        config_actual.clear()
        config_actual.update(historial_experimentos[i][1])
        for n in nombres[i:]:
            decisiones.pop(n, None)
        del historial_experimentos[i:]
        del resultados_experimentos[i:]
        print(f"[{nombre}] Experimento repetido: la configuración vuelve a su estado previo "
              f"y se descartan las decisiones posteriores ({', '.join(nombres[i:])}).")
    historial_experimentos.append((nombre, dict(config_actual)))
    historias_ultimo_experimento.clear()
    tabla = armar_tabla(entrenar_variantes(nombre, variantes, config_actual, forzar))
    resultados_experimentos.append(tabla)
    return tabla


def ampliar_rango(nombre, variantes, config_actual, hiperparametro, arriba, abajo, minimo, maximo):
    tabla = resultados_experimentos[-1]
    while True:
        valores = {etq: cambios[hiperparametro] for etq, cambios in variantes.items()}
        mejor = valores[tabla.loc[0, "variante"]]
        if mejor == max(valores.values()) and arriba(mejor) <= maximo:
            nuevo = arriba(mejor)
        elif mejor == min(valores.values()) and abajo(mejor) >= minimo:
            nuevo = abajo(mejor)
        else:
            return tabla
        etiqueta = f"{nuevo:g}" if isinstance(nuevo, float) else str(nuevo)
        print(f"[{nombre}] La mejor variante está en el borde del rango: se agrega {hiperparametro} = {etiqueta}")
        variantes[etiqueta] = {hiperparametro: nuevo}
        filas = tabla.drop(columns=["diferencia_vs_mejor", "dentro_del_margen"]).to_dict("records")
        filas += entrenar_variantes(nombre, {etiqueta: {hiperparametro: nuevo}}, config_actual)
        tabla = armar_tabla(filas)
        resultados_experimentos[-1] = tabla


def seleccionar_variante(tabla):
    mejor = tabla.loc[0, "variante"]
    empatadas = tabla.loc[tabla["dentro_del_margen"] & (tabla["variante"] != mejor), "variante"].tolist()
    print(f"Mejor F1 macro de validación: {mejor}")
    if empatadas:
        print(f"En empate técnico (dentro de {MARGEN_ERROR * 100:.2f} pp): {', '.join(map(str, empatadas))}")
    return mejor


def tabla_de(nombre):
    return resultados_experimentos[[n for n, _ in historial_experimentos].index(nombre)]


def aplicar_variante(nombre, variantes, elegida):
    tabla, config_previa = resultados_experimentos[-1], historial_experimentos[-1][1]
    previa = [etq for etq, cambios in variantes.items() if all(config_previa.get(k) == v for k, v in cambios.items())]
    f1_de = dict(zip(tabla["variante"], tabla["f1_macro_val"]))
    decisiones[nombre] = {"elegida": elegida, "f1_elegida": f1_de[elegida],
                          "variante_previa": previa[0] if previa else None,
                          "f1_previa": f1_de[previa[0]] if previa else None}
    CONFIG_ACTUAL.update(variantes[elegida])
    mostrar_config(CONFIG_ACTUAL)

### 5.4 Curvas de entrenamiento y configuración

- `graficar_curvas(historia, titulo)`: pérdida, exactitud y F1 macro de validación por época en tres paneles horizontales, con la mejor época marcada.
- `graficar_comparacion(historias, clave, titulo, eje_y)`: una línea por variante para la métrica `clave` (por ejemplo, `f1_val`), para comparar cómo aprende cada una.
- `mostrar_config(config)`: muestra la configuración como tabla vertical.

In [ ]:
# ============================================================
# Visualización de curvas y configuración
# ============================================================
def graficar_curvas(historia, titulo):
    epocas = list(range(1, len(historia["loss_train"]) + 1))
    mejor = int(np.argmin(historia["loss_val"])) + 1
    fig = make_subplots(rows=1, cols=3, subplot_titles=["Pérdida", "Exactitud", "F1 macro (validación)"])
    for col, tr, va in [(1, "loss_train", "loss_val"), (2, "exactitud_train", "exactitud_val")]:
        fig.add_trace(go.Scatter(x=epocas, y=historia[tr], name="entrenamiento", line_color=COLORES[2],
                                 legendgroup="train", showlegend=col == 1), row=1, col=col)
        fig.add_trace(go.Scatter(x=epocas, y=historia[va], name="validación", line_color=COLORES[0],
                                 legendgroup="val", showlegend=col == 1), row=1, col=col)
    fig.add_trace(go.Scatter(x=epocas, y=historia["f1_val"], name="F1 macro validación", line_color=COLORES[1]), row=1, col=3)
    for col in [1, 2, 3]:
        fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, row=1, col=col)
        fig.update_xaxes(title_text="época", row=1, col=col)
    fig.update_layout(title=f"{titulo} · mejor época: {mejor}", height=420)
    fig.show()


def graficar_comparacion(historias, clave, titulo, eje_y):
    paleta = COLORES + ["#0E2747", "#8A6A00", "#52514E"]
    fig = go.Figure()
    for i, (etiqueta, historia) in enumerate(historias.items()):
        fig.add_trace(go.Scatter(x=list(range(1, len(historia[clave]) + 1)), y=historia[clave], name=etiqueta,
                                 line_color=paleta[i % len(paleta)]))
    fig.update_layout(title=titulo, xaxis_title="época", yaxis_title=eje_y, height=420)
    fig.show()


def mostrar_config(config):
    display(pd.DataFrame({"valor": pd.Series(config).astype(str)}))

## 6. Arquitectura

### 6.1 Red convolucional configurable y MLP de comparación

**CNN (`ClasificadorCNN`).** Se compone de `bloques` bloques convolucionales seguidos de una capa densa:

- **Bloque convolucional:** una convolución con `filtros × 2^i` filtros de tamaño `kernel × kernel` (con relleno para conservar el tamaño), *batch normalization* opcional, la función de activación y un *max pooling* de 2 × 2 que reduce a la mitad el alto y el ancho. Con 2 bloques, la imagen pasa de 28 × 28 a 7 × 7; duplicar los filtros en cada bloque compensa la pérdida de resolución con más canales.
- **Capa densa (representación):** aplana los mapas de características y los proyecta a `unidades_densas` neuronas con la misma activación. Este vector es la **representación visual** que se extrae para la fusión.
- **Salida:** una capa lineal de 10 neuronas (una por clase). No lleva *softmax*, porque la entropía cruzada de PyTorch lo aplica internamente.
- **Dropout** antes de la capa densa y antes de la salida, con la tasa `dropout`.

`representacion(x)` retorna el vector de la capa densa y `forward(x)` retorna las 10 salidas (*logits*).

**MLP (`ClasificadorMLP`).** Aplana la imagen en 784 valores y aplica `capas_mlp` capas densas de `unidades_mlp` neuronas, cada una con activación y Dropout, antes de la salida. Sirve para medir cuánto aporta la estructura convolucional con un número de parámetros parecido.

`ACTIVACIONES` reúne las funciones de activación que se comparan en la sección 11 y `construir_modelo(config)` crea la red indicada por `config["tipo"]` en el dispositivo.

In [ ]:
# ============================================================
# Definición de la CNN, del MLP y del constructor
# ============================================================
ACTIVACIONES = {"relu": nn.ReLU, "leaky_relu": lambda: nn.LeakyReLU(0.01), "gelu": nn.GELU,
                "tanh": nn.Tanh, "sigmoide": nn.Sigmoid}


class ClasificadorCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        capas, canales, lado = [], 1, 28
        for i in range(config["bloques"]):
            salida = config["filtros"] * 2 ** i
            capas.append(nn.Conv2d(canales, salida, config["kernel"], padding=config["kernel"] // 2))
            if config["batchnorm"]:
                capas.append(nn.BatchNorm2d(salida))
            capas += [ACTIVACIONES[config["activacion"]](), nn.MaxPool2d(2)]
            canales, lado = salida, lado // 2
        self.extractor = nn.Sequential(*capas, nn.Flatten())
        self.densa = nn.Sequential(nn.Dropout(config["dropout"]),
                                   nn.Linear(canales * lado * lado, config["unidades_densas"]),
                                   ACTIVACIONES[config["activacion"]]())
        self.salida = nn.Sequential(nn.Dropout(config["dropout"]), nn.Linear(config["unidades_densas"], 10))

    def representacion(self, x):
        return self.densa(self.extractor(x))

    def forward(self, x):
        return self.salida(self.representacion(x))


class ClasificadorMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        capas, entrada = [nn.Flatten()], 28 * 28
        for _ in range(config["capas_mlp"]):
            capas += [nn.Linear(entrada, config["unidades_mlp"]), ACTIVACIONES[config["activacion"]](),
                      nn.Dropout(config["dropout"])]
            entrada = config["unidades_mlp"]
        self.ocultas = nn.Sequential(*capas)
        self.salida = nn.Linear(entrada, 10)

    def representacion(self, x):
        return self.ocultas(x)

    def forward(self, x):
        return self.salida(self.representacion(x))


def construir_modelo(config):
    clase = ClasificadorCNN if config["tipo"] == "cnn" else ClasificadorMLP
    return clase(config).to(device)

### 6.2 Configuración base

| Hiperparámetro | Valor base | Justificación |
|---|---|---|
| `tipo` | `cnn` | Arquitectura pedida para imágenes; el MLP solo se usa en la comparación de la sección 9 |
| `bloques` | 2 | Lleva la imagen de 28 × 28 a 7 × 7: conserva la silueta de la prenda con un costo bajo |
| `filtros` | 32 (64 en el segundo bloque) | Valor habitual en redes pequeñas; duplicar los filtros al reducir la resolución sigue el diseño de VGG (Simonyan y Zisserman, 2015) |
| `kernel` | 3 | El tamaño más pequeño que capta un vecindario completo; dos convoluciones de 3 × 3 cubren el mismo campo que una de 5 × 5 con menos parámetros |
| `activacion` | `relu` | No se satura para valores positivos, lo que evita el desvanecimiento del gradiente de la sigmoide y la tangente hiperbólica |
| `batchnorm` | `False` | Base simple; su efecto se mide por separado |
| `unidades_densas` | 128 | Representación compacta para la fusión: 24 veces más pequeña que los 3.136 valores del último mapa |
| `dropout` | 0,25 | Regularización moderada para una red de ≈ 420.000 parámetros entrenada con 54.000 imágenes |
| `capas_mlp`, `unidades_mlp` | 2, 256 | Solo para el MLP de comparación |
| `normalizacion` | `escala_0_1` | La forma más simple que evita entradas de 0 a 255 |
| `aumento` | `False` | Base sin aumento; su efecto se mide por separado |
| `perdida` | `entropia_cruzada` | Función estándar para clasificación multiclase |
| `optimizador` | `adam` | Ajusta la tasa por parámetro y converge sin una búsqueda fina de la tasa (Kingma y Ba, 2015) |
| `learning_rate` | 0,001 | Valor por defecto propuesto para Adam por sus autores |
| `weight_decay` | 0 | Base sin penalización L2; su efecto se mide por separado |
| `batch_size` | 128 | 422 actualizaciones por época con 54.000 imágenes: suficientes para converger en pocas épocas sin que cada época sea lenta |

In [ ]:
# ============================================================
# Configuración base
# ============================================================
CONFIG_BASE = dict(tipo="cnn", bloques=2, filtros=32, kernel=3, activacion="relu", batchnorm=False,
                   unidades_densas=128, dropout=0.25, capas_mlp=2, unidades_mlp=256,
                   normalizacion="escala_0_1", aumento=False, perdida="entropia_cruzada",
                   optimizador="adam", learning_rate=0.001, weight_decay=0.0, batch_size=128)
CONFIG_ACTUAL = dict(CONFIG_BASE)
mostrar_config(CONFIG_BASE)

## 7. Entrenamiento del modelo base

Se entrena la configuración base con la semilla 42, mostrando el avance por época. Esta corrida es la referencia de todos los experimentos y la primera de las 5 semillas del rango de error.

In [ ]:
# ============================================================
# Entrenamiento del modelo base
# ============================================================
fila_base, historia_base, modelo_base = entrenar_y_registrar(CONFIG_BASE, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Estructura y parámetros del modelo base
# ============================================================
print(modelo_base)
print(f"Parámetros entrenables: {contar_parametros(modelo_base):,}")

In [ ]:
# ============================================================
# Métricas del modelo base en su mejor época
# ============================================================
display(pd.DataFrame([fila_base]).round(4))

In [ ]:
# ============================================================
# Curvas del modelo base
# ============================================================
graficar_curvas(historia_base, "Modelo base")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro y exactitud de validación frente a las líneas base (sección 3.7).
- Mejor época, brecha entrenamiento–validación y si la pérdida de validación empieza a subir (sobreajuste).

## 8. Rango de error entre reentrenamientos

Entrenar dos veces la misma configuración no entrega el mismo resultado: cambian los pesos iniciales, el orden de los lotes y el aumento aleatorio. Si no se conoce el tamaño de esa variación, una diferencia pequeña entre dos configuraciones podría interpretarse como mejora cuando es ruido.

Se reentrena `CONFIG_BASE` con otras cuatro semillas (5 en total) y se calculan la media, la desviación estándar, el mínimo, el máximo y el **rango** (máximo − mínimo) del F1 macro de validación. Ese rango se guarda en `MARGEN_ERROR` y se usa en todo el notebook: dos variantes cuya diferencia sea menor o igual se consideran en **empate técnico**.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración base con distintas semillas
# ============================================================
SEMILLAS_ADICIONALES = [1, 2, 3, 4]

filas_variabilidad = [{"semilla": SEED, **fila_base}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_BASE, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_variabilidad.append({"semilla": semilla, **fila})
tabla_variabilidad = pd.DataFrame(filas_variabilidad)

In [ ]:
# ============================================================
# Resultados por semilla
# ============================================================
display(tabla_variabilidad.round(4))

In [ ]:
# ============================================================
# Estadísticos de variación y rango de error
# ============================================================
columnas_variacion = ["f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val", "mejor_epoca"]
estadisticos_variabilidad = tabla_variabilidad[columnas_variacion].agg(["mean", "std", "min", "max"]).T
estadisticos_variabilidad["rango"] = estadisticos_variabilidad["max"] - estadisticos_variabilidad["min"]
MARGEN_ERROR = estadisticos_variabilidad.loc["f1_macro_val", "rango"]
print(f"Rango de error entre reentrenamientos (MARGEN_ERROR): {MARGEN_ERROR:.4f} ({MARGEN_ERROR * 100:.2f} pp)")
display(estadisticos_variabilidad.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Media, desviación estándar y rango del F1 macro con 5 semillas: el valor de MARGEN_ERROR en puntos porcentuales.
- Variación de la mejor época entre semillas.

## 9. Experimento clave: CNN frente a MLP

Antes de ajustar la CNN se comprueba con evidencia propia que la estructura convolucional aporta. Se comparan la CNN base (≈ 420.000 parámetros) y dos MLP de dos capas ocultas con 256 y 512 neuronas (≈ 270.000 y ≈ 670.000 parámetros), que encierran el tamaño de la CNN. Todos usan la misma normalización, función de pérdida, optimizador, tasa, lote, Dropout y régimen de épocas.

Si la CNN supera a ambos MLP por más que el rango de error, la ventaja se debe a su estructura (filtros locales y pesos compartidos) y no al número de parámetros. La CNN se mantiene en cualquier caso, porque el sistema pide una representación convolucional; el experimento mide cuánto aporta.

### 9.1 Comparación de arquitecturas

Variantes: CNN base, MLP 2 × 256 y MLP 2 × 512.

In [ ]:
# ============================================================
# Experimento 9.1: comparación de arquitecturas
# ============================================================
variantes_arquitectura = {"CNN (2 bloques)": dict(tipo="cnn"), "MLP 2×256": dict(tipo="mlp", unidades_mlp=256), "MLP 2×512": dict(tipo="mlp", unidades_mlp=512)}
tabla_arquitectura = ejecutar_experimento("arquitectura", variantes_arquitectura, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 9.1
# ============================================================
display(tabla_arquitectura[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 9.1
# ============================================================
elegida = "CNN (2 bloques)"  # requerida por el enunciado: el experimento aporta evidencia, no decide
aplicar_variante("arquitectura", variantes_arquitectura, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Diferencia de F1 macro entre la CNN y el mejor MLP, comparada con MARGEN_ERROR.
- Parámetros de cada modelo: ¿la ventaja se explica por tamaño o por estructura?
- Brecha entrenamiento–validación de cada uno.

## 10. Experimentos de preprocesamiento

Se mide el efecto en el desempeño de los dos ajustes de preprocesamiento de la sección 4: la escala de los píxeles y el aumento de datos.

### 10.1 Normalización de los píxeles

Se comparan las tres escalas de la sección 4.1. Se espera que los valores sin escalar entrenen peor con la misma tasa de aprendizaje, porque las activaciones de la primera capa son 255 veces mayores, y que la escala [0, 1] y la estandarizada queden cerca entre sí.

In [ ]:
# ============================================================
# Experimento 10.1: normalización de los píxeles
# ============================================================
variantes_normalizacion = {"sin escalar (0–255)": dict(normalizacion="ninguna"), "escala [0, 1]": dict(normalizacion="escala_0_1"), "estandarizada": dict(normalizacion="estandar")}
tabla_normalizacion = ejecutar_experimento("normalización", variantes_normalizacion, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 10.1
# ============================================================
display(tabla_normalizacion[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 10.1
# ============================================================
elegida = seleccionar_variante(tabla_normalizacion)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("normalización", variantes_normalizacion, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 10.2 Aumento de datos

Se compara el entrenamiento sin aumento con el aumento de la sección 4.2 (volteo horizontal y desplazamiento de hasta 2 píxeles). El aumento suele reducir la brecha entre entrenamiento y validación, pero también hace más difícil cada época, por lo que puede necesitar más épocas: el régimen de extensión evita compararlo antes de que converja.

In [ ]:
# ============================================================
# Experimento 10.2: aumento de datos
# ============================================================
variantes_aumento = {"sin aumento": dict(aumento=False), "volteo + desplazamiento": dict(aumento=True)}
tabla_aumento = ejecutar_experimento("aumento", variantes_aumento, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 10.2
# ============================================================
display(tabla_aumento[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 10.2
# ============================================================
elegida = seleccionar_variante(tabla_aumento)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("aumento", variantes_aumento, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 11. Experimentos de funciones de activación y de pérdida

Se comparan las funciones que definen cómo se transforma la señal dentro de la red (activación) y qué se minimiza al entrenar (pérdida). Estas corridas se entrenan siempre, para graficar sus curvas y mostrar cómo cambia el aprendizaje, no solo el resultado final.

### 11.1 Función de activación

- **ReLU** `max(0, x)`: simple y sin saturación para valores positivos; las neuronas con entrada siempre negativa dejan de aprender.
- **Leaky ReLU**: como ReLU, pero con una pendiente de 0,01 para valores negativos, de modo que ninguna neurona queda sin gradiente.
- **GELU**: versión suave de ReLU que pondera cada valor por su probabilidad bajo una normal (Hendrycks y Gimpel, 2016).
- **Tangente hiperbólica** y **sigmoide**: se saturan en los extremos (gradiente cercano a 0), lo que hace más lento el aprendizaje en redes con varias capas.

Se espera que las tres primeras queden cerca y que la sigmoide aprenda más lento.

In [ ]:
# ============================================================
# Experimento 11.1: función de activación
# ============================================================
variantes_activacion = {"ReLU": dict(activacion="relu"), "Leaky ReLU": dict(activacion="leaky_relu"), "GELU": dict(activacion="gelu"), "Tanh": dict(activacion="tanh"), "Sigmoide": dict(activacion="sigmoide")}
tabla_activacion = ejecutar_experimento("activación", variantes_activacion, CONFIG_ACTUAL, forzar=True)
historias_activacion = dict(historias_ultimo_experimento)

In [ ]:
# ============================================================
# Resultados del experimento 11.1
# ============================================================
display(tabla_activacion[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Curvas de validación del experimento 11.1
# ============================================================
graficar_comparacion(historias_activacion, "f1_val", "F1 macro de validación por época según la activación", "F1 macro de validación")

In [ ]:
# ============================================================
# Selección de la variante del experimento 11.1
# ============================================================
elegida = seleccionar_variante(tabla_activacion)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("activación", variantes_activacion, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 11.2 Función de pérdida

- **Entropía cruzada:** penaliza el logaritmo de la probabilidad asignada a la clase correcta; el gradiente es grande cuando el modelo se equivoca con confianza.
- **Entropía cruzada con suavizado de etiquetas (0,1):** evita que el modelo empuje las probabilidades a 0 y 1, lo que reduce la sobreconfianza (Szegedy et al., 2016).
- **Error cuadrático medio (MSE)** entre las probabilidades y la etiqueta *one-hot*: su gradiente se achica cuando el *softmax* se satura, aunque la predicción sea incorrecta, por lo que se espera un aprendizaje más lento.

La pérdida de validación de la tabla se mide siempre con entropía cruzada, para que sea comparable entre las tres variantes.

In [ ]:
# ============================================================
# Experimento 11.2: función de pérdida
# ============================================================
variantes_perdida = {"entropía cruzada": dict(perdida="entropia_cruzada"), "suavizado de etiquetas": dict(perdida="suavizado_etiquetas"), "MSE": dict(perdida="mse")}
tabla_perdida = ejecutar_experimento("pérdida", variantes_perdida, CONFIG_ACTUAL, forzar=True)
historias_perdida = dict(historias_ultimo_experimento)

In [ ]:
# ============================================================
# Resultados del experimento 11.2
# ============================================================
display(tabla_perdida[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Curvas de validación del experimento 11.2
# ============================================================
graficar_comparacion(historias_perdida, "f1_val", "F1 macro de validación por época según la función de pérdida", "F1 macro de validación")

In [ ]:
# ============================================================
# Selección de la variante del experimento 11.2
# ============================================================
elegida = seleccionar_variante(tabla_perdida)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("pérdida", variantes_perdida, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 12. Experimentos de entrenamiento: tasa de aprendizaje, tamaño del lote, optimizador y épocas

Son los tres hiperparámetros que pide optimizar el problema (tasa, lote y épocas), más el optimizador, que define cómo se usa la tasa.

### 12.1 Tasa de aprendizaje

La tasa define el tamaño de cada paso de actualización. Con una tasa muy baja el modelo avanza lento y puede no converger dentro del tope de épocas; con una muy alta, la pérdida oscila o diverge. Se prueban 0,0003, 0,001 y 0,003 (pasos de ≈ 3 veces); si la mejor queda en un borde, se amplía hasta 0,0001 o 0,01.

In [ ]:
# ============================================================
# Experimento 12.1: tasa de aprendizaje
# ============================================================
variantes_tasa = {"0.0003": dict(learning_rate=0.0003), "0.001": dict(learning_rate=0.001), "0.003": dict(learning_rate=0.003)}
tabla_tasa = ejecutar_experimento("tasa de aprendizaje", variantes_tasa, CONFIG_ACTUAL)
tabla_tasa = ampliar_rango("tasa de aprendizaje", variantes_tasa, CONFIG_ACTUAL, "learning_rate", lambda v: round(v * 3, 6), lambda v: round(v / 3, 6), 0.0001, 0.01)

In [ ]:
# ============================================================
# Resultados del experimento 12.1
# ============================================================
display(tabla_tasa[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 12.1
# ============================================================
elegida = seleccionar_variante(tabla_tasa)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("tasa de aprendizaje", variantes_tasa, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 12.2 Tamaño del lote

Con 54.000 imágenes, un lote de 64 da 844 actualizaciones por época; uno de 256, 211. Los lotes pequeños actualizan más veces y con más ruido (que puede ayudar a generalizar), pero aprovechan peor la GPU; los grandes son más rápidos por época, pero pueden necesitar una tasa mayor. Se prueban 64, 128 y 256 con la tasa vigente; si la mejor queda en un borde, se amplía entre 16 y 1.024. La interacción entre lote y tasa se verifica en la sección 19.

In [ ]:
# ============================================================
# Experimento 12.2: tamaño del lote
# ============================================================
variantes_lote = {"64": dict(batch_size=64), "128": dict(batch_size=128), "256": dict(batch_size=256)}
tabla_lote = ejecutar_experimento("tamaño del lote", variantes_lote, CONFIG_ACTUAL)
tabla_lote = ampliar_rango("tamaño del lote", variantes_lote, CONFIG_ACTUAL, "batch_size", lambda v: v * 2, lambda v: v // 2, 16, 1024)

In [ ]:
# ============================================================
# Resultados del experimento 12.2
# ============================================================
display(tabla_lote[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 12.2
# ============================================================
elegida = seleccionar_variante(tabla_lote)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("tamaño del lote", variantes_lote, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 12.3 Optimizador

Se compara Adam (con la tasa vigente) con SGD con *momentum* 0,9. SGD necesita tasas más altas que Adam, así que se prueba con su propia tasa en dos valores (0,01 y 0,05) para que la comparación sea justa. Si gana una variante de SGD, la configuración adopta también su tasa.

In [ ]:
# ============================================================
# Experimento 12.3: optimizador
# ============================================================
variantes_optimizador = {"Adam (tasa vigente)": dict(optimizador="adam"), "SGD momentum, tasa 0,01": dict(optimizador="sgd", learning_rate=0.01), "SGD momentum, tasa 0,05": dict(optimizador="sgd", learning_rate=0.05)}
tabla_optimizador = ejecutar_experimento("optimizador", variantes_optimizador, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 12.3
# ============================================================
display(tabla_optimizador[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 12.3
# ============================================================
elegida = seleccionar_variante(tabla_optimizador)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("optimizador", variantes_optimizador, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 12.4 Épocas sin Early Stopping

Para justificar el tope de 20 épocas y la paciencia de 3, se entrena la configuración vigente durante 40 épocas **sin Early Stopping** y se observa en qué época la pérdida de validación deja de bajar y empieza a subir (inicio del sobreajuste). Esta corrida no cambia la configuración: muestra si el régimen de épocas usado en todos los experimentos es suficiente.

In [ ]:
# ============================================================
# Experimento 12.4: 40 épocas sin Early Stopping
# ============================================================
EPOCAS_SIN_ES = 40
fila_epocas, historia_epocas, _ = entrenar_y_registrar(CONFIG_ACTUAL, epocas=EPOCAS_SIN_ES, paciencia=0, forzar=True)

In [ ]:
# ============================================================
# Curvas sin Early Stopping
# ============================================================
graficar_curvas(historia_epocas, "Configuración vigente, 40 épocas sin Early Stopping")

In [ ]:
# ============================================================
# Métricas de validación en épocas seleccionadas
# ============================================================
epocas_revisadas = [5, 10, 15, 20, 30, 40]
tabla_epocas = pd.DataFrame({k: [historia_epocas[k][e - 1] for e in epocas_revisadas]
                             for k in ["loss_train", "loss_val", "exactitud_val", "f1_val"]}, index=epocas_revisadas)
tabla_epocas.index.name = "época"
print(f"Mejor época por pérdida de validación: {int(np.argmin(historia_epocas['loss_val'])) + 1}")
display(tabla_epocas.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Época de menor pérdida de validación y desde cuándo sube (sobreajuste).
- ¿El F1 de validación sigue mejorando aunque la pérdida suba? (la pérdida castiga la sobreconfianza).
- ¿El tope de 20 épocas con paciencia 3 es suficiente para esta configuración?

## 13. Experimentos de arquitectura

Se ajusta la capacidad de la CNN: profundidad, número y tamaño de los filtros, normalización por lotes y tamaño de la representación.

### 13.1 Número de bloques convolucionales

Con 1 bloque, la capa densa recibe mapas de 14 × 14 y la red ve solo patrones muy locales; con 3, los mapas quedan en 3 × 3 y cada filtro abarca casi toda la prenda. No se prueba un cuarto bloque: con 28 × 28 píxeles, un cuarto *pooling* dejaría mapas de 1 × 1, sin estructura espacial que aprovechar.

In [ ]:
# ============================================================
# Experimento 13.1: número de bloques convolucionales
# ============================================================
variantes_bloques = {"1": dict(bloques=1), "2": dict(bloques=2), "3": dict(bloques=3)}
tabla_bloques = ejecutar_experimento("bloques", variantes_bloques, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 13.1
# ============================================================
display(tabla_bloques[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.1
# ============================================================
elegida = seleccionar_variante(tabla_bloques)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("bloques", variantes_bloques, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 13.2 Número de filtros del primer bloque

Más filtros permiten detectar más patrones distintos, pero multiplican los parámetros y el tiempo por época. Se prueban 16, 32 y 64 filtros en el primer bloque (el resto se duplica en cada bloque); si gana 64, se amplía a 128.

In [ ]:
# ============================================================
# Experimento 13.2: número de filtros del primer bloque
# ============================================================
variantes_filtros = {"16": dict(filtros=16), "32": dict(filtros=32), "64": dict(filtros=64)}
tabla_filtros = ejecutar_experimento("filtros", variantes_filtros, CONFIG_ACTUAL)
tabla_filtros = ampliar_rango("filtros", variantes_filtros, CONFIG_ACTUAL, "filtros", lambda v: v * 2, lambda v: v // 2, 16, 128)

In [ ]:
# ============================================================
# Resultados del experimento 13.2
# ============================================================
display(tabla_filtros[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.2
# ============================================================
elegida = seleccionar_variante(tabla_filtros)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("filtros", variantes_filtros, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 13.3 Tamaño del kernel

Un kernel de 5 × 5 ve un vecindario más amplio en cada capa que uno de 3 × 3, a cambio de casi el triple de parámetros por filtro. Se prueban 3 y 5; si gana 5, se amplía a 7.

In [ ]:
# ============================================================
# Experimento 13.3: tamaño del kernel
# ============================================================
variantes_kernel = {"3": dict(kernel=3), "5": dict(kernel=5)}
tabla_kernel = ejecutar_experimento("kernel", variantes_kernel, CONFIG_ACTUAL)
tabla_kernel = ampliar_rango("kernel", variantes_kernel, CONFIG_ACTUAL, "kernel", lambda v: v + 2, lambda v: v - 2, 3, 7)

In [ ]:
# ============================================================
# Resultados del experimento 13.3
# ============================================================
display(tabla_kernel[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.3
# ============================================================
elegida = seleccionar_variante(tabla_kernel)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("kernel", variantes_kernel, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 13.4 Batch normalization

La *batch normalization* normaliza las activaciones de cada canal dentro del lote, lo que estabiliza el entrenamiento y permite tasas más altas (Ioffe y Szegedy, 2015). En combinación con ciertas tasas de aprendizaje puede inestabilizar el entrenamiento; si una variante colapsa (exactitud cercana al 10 %), se registra como **Error / Falla** y se diagnostica.

In [ ]:
# ============================================================
# Experimento 13.4: batch normalization
# ============================================================
variantes_batchnorm = {"sin batch norm": dict(batchnorm=False), "con batch norm": dict(batchnorm=True)}
tabla_batchnorm = ejecutar_experimento("batch norm", variantes_batchnorm, CONFIG_ACTUAL)

In [ ]:
# ============================================================
# Resultados del experimento 13.4
# ============================================================
display(tabla_batchnorm[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.4
# ============================================================
elegida = seleccionar_variante(tabla_batchnorm)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("batch norm", variantes_batchnorm, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 13.5 Unidades de la capa densa (dimensión de la representación)

Esta capa define el tamaño del vector que la CNN entrega a la fusión. Una representación más grande conserva más información visual, pero agrega parámetros al clasificador fusionado. Se prueban 64, 128 y 256 unidades; si la mejor queda en un borde, se amplía entre 32 y 512.

In [ ]:
# ============================================================
# Experimento 13.5: unidades de la capa densa (dimensión de la representación)
# ============================================================
variantes_unidades = {"64": dict(unidades_densas=64), "128": dict(unidades_densas=128), "256": dict(unidades_densas=256)}
tabla_unidades = ejecutar_experimento("unidades densas", variantes_unidades, CONFIG_ACTUAL)
tabla_unidades = ampliar_rango("unidades densas", variantes_unidades, CONFIG_ACTUAL, "unidades_densas", lambda v: v * 2, lambda v: v // 2, 32, 512)

In [ ]:
# ============================================================
# Resultados del experimento 13.5
# ============================================================
display(tabla_unidades[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 13.5
# ============================================================
elegida = seleccionar_variante(tabla_unidades)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("unidades densas", variantes_unidades, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 14. Experimentos de regularización

Se prueban por separado dos técnicas de regularización, cada una con al menos dos valores distintos de cero, para aislar el efecto de cada una.

### 14.1 Dropout

El Dropout apaga al azar una fracción de las neuronas en cada paso de entrenamiento, lo que obliga a la red a no depender de neuronas individuales (Srivastava et al., 2014). Se prueban 0; 0,25 y 0,5; si gana 0,5, se amplía a 0,75.

In [ ]:
# ============================================================
# Experimento 14.1: dropout
# ============================================================
variantes_dropout = {"0.0": dict(dropout=0.0), "0.25": dict(dropout=0.25), "0.5": dict(dropout=0.5)}
tabla_dropout = ejecutar_experimento("dropout", variantes_dropout, CONFIG_ACTUAL)
tabla_dropout = ampliar_rango("dropout", variantes_dropout, CONFIG_ACTUAL, "dropout", lambda v: v + 0.25, lambda v: v - 0.25, 0.0, 0.75)

In [ ]:
# ============================================================
# Resultados del experimento 14.1
# ============================================================
display(tabla_dropout[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 14.1
# ============================================================
elegida = seleccionar_variante(tabla_dropout)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("dropout", variantes_dropout, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

### 14.2 Weight decay (penalización L2)

El *weight decay* resta en cada paso una fracción de los pesos, lo que favorece pesos pequeños y modelos más suaves. Se prueban 0; 0,00001 y 0,0001; si gana el mayor, se amplía a 0,001.

In [ ]:
# ============================================================
# Experimento 14.2: weight decay (penalización l2)
# ============================================================
variantes_weight_decay = {"0": dict(weight_decay=0.0), "1e-05": dict(weight_decay=1e-5), "0.0001": dict(weight_decay=1e-4)}
tabla_weight_decay = ejecutar_experimento("weight decay", variantes_weight_decay, CONFIG_ACTUAL)
tabla_weight_decay = ampliar_rango("weight decay", variantes_weight_decay, CONFIG_ACTUAL, "weight_decay", lambda v: v * 10, lambda v: v / 10 if v > 1e-5 else -1.0, 0.0, 0.001)

In [ ]:
# ============================================================
# Resultados del experimento 14.2
# ============================================================
display(tabla_weight_decay[COLUMNAS_TABLA].round(4))

In [ ]:
# ============================================================
# Selección de la variante del experimento 14.2
# ============================================================
elegida = seleccionar_variante(tabla_weight_decay)
# elegida = "<etiqueta>"  # Cambio Manual
aplicar_variante("weight decay", variantes_weight_decay, elegida)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Variante ganadora y diferencia con la segunda en puntos porcentuales; ¿supera MARGEN_ERROR?
- Costo (parámetros, segundos por época) de las variantes empatadas.
- Si hay empate técnico, opciones y criterio de decisión (se decide con el usuario).

## 15. Modelo final

### 15.1 Configuración final y tabla de decisiones

La configuración final es la que quedó en `CONFIG_ACTUAL` después de todos los experimentos. La tabla de decisiones muestra, para cada experimento, la variante elegida, su F1 macro de validación y el de la variante que coincidía con la configuración previa, para ver cuánto aportó cada decisión y si supera el rango de error.

In [ ]:
# ============================================================
# Configuración final
# ============================================================
CONFIG_FINAL = dict(CONFIG_ACTUAL)
mostrar_config(CONFIG_FINAL)

In [ ]:
# ============================================================
# Tabla de decisiones
# ============================================================
tabla_decisiones = pd.DataFrame.from_dict(decisiones, orient="index")
tabla_decisiones.index.name = "experimento"
tabla_decisiones["aporte"] = tabla_decisiones["f1_elegida"] - tabla_decisiones["f1_previa"]
tabla_decisiones["supera_margen"] = tabla_decisiones["aporte"].abs() > MARGEN_ERROR
tabla_decisiones = tabla_decisiones.reset_index()
display(tabla_decisiones.round(4))

### 15.2 Entrenamiento del modelo final

Se entrena la configuración final con la semilla 42 y el mismo régimen de épocas que los experimentos. Esta corrida se entrena siempre, porque su modelo se evalúa en test y se guarda para la fusión.

In [ ]:
# ============================================================
# Entrenamiento del modelo final
# ============================================================
fila_final, historia_final, modelo_final = entrenar_y_registrar(CONFIG_FINAL, verbose=True, forzar=True)

In [ ]:
# ============================================================
# Métricas del modelo final en su mejor época
# ============================================================
print(f"Parámetros entrenables: {contar_parametros(modelo_final):,}")
display(pd.DataFrame([fila_final]).round(4))

### 15.3 Curvas de entrenamiento del modelo final

Curvas de pérdida, exactitud y F1 macro de validación del modelo final, con la mejor época marcada.

In [ ]:
# ============================================================
# Curvas del modelo final
# ============================================================
graficar_curvas(historia_final, "Modelo final")

**Observación / Hallazgo:** *(completar después de ejecutar)*

- F1 macro de validación del modelo final frente al base y frente a la media de las 5 semillas del base.
- Mejor época y brecha entrenamiento–validación frente al base.

## 16. Evaluación del modelo final en el conjunto de test

El conjunto de test (10.000 imágenes, 1.000 por clase) no se usó en ninguna decisión. Se evalúa una sola vez con el modelo final. Qué indica cada métrica en este problema:

- **Exactitud:** proporción de prendas bien clasificadas. Con clases balanceadas es una medida directa del desempeño global.
- **Precisión por clase:** de las imágenes que el modelo llama «camisa», cuántas lo son. Una precisión baja indica que el modelo **sobrepredice** esa clase.
- **Exhaustividad por clase:** de las camisas reales, cuántas detecta. Una exhaustividad baja indica que esa clase se confunde con otras.
- **F1 macro:** promedio del F1 de las 10 clases; baja si alguna clase se resuelve mal aunque el resto esté bien.

Una clase solo se califica como bien resuelta si su precisión y su exhaustividad son altas a la vez.

### 16.1 Métricas globales frente a las líneas base

Se comparan las métricas del modelo final en test con las dos líneas base, calculadas también en test.

In [ ]:
# ============================================================
# Predicción en test y métricas globales
# ============================================================
pred_test = predecir(modelo_final, X_TEST, Y_TEST, CONFIG_FINAL, CRITERIO_EVAL)
pred_val = predecir(modelo_final, X_VAL, Y_VAL, CONFIG_FINAL, CRITERIO_EVAL)
tabla_test = pd.DataFrame([
    {"modelo": "Clase mayoritaria", **metricas(y_test, np.full(len(y_test), CLASE_MAYORITARIA))},
    {"modelo": "Prototipo medio por clase", **metricas(y_test, predecir_prototipo(x_test, PROTOTIPOS))},
    {"modelo": "CNN final", **metricas(pred_test["y_true"], pred_test["y_pred"]), "loss": pred_test["perdida"]},
])
display(tabla_test.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Exactitud y F1 macro de la CNN en test frente a las líneas base (diferencia en puntos porcentuales).
- Diferencia entre validación y test: ¿el test confirma lo estimado en validación?

### 16.2 Métricas por clase

Se muestran la precisión, la exhaustividad y el F1 de cada clase en test y, además, en validación. Con 1.000 imágenes por clase en test, el reporte de validación (600 por clase) sirve como segunda medición independiente para no sacar conclusiones por clase de un solo conjunto.

`reporte_por_clase(y_true, y_pred)` retorna una tabla con precisión, exhaustividad, F1 y soporte por clase.

In [ ]:
# ============================================================
# Reporte por clase en test
# ============================================================
def reporte_por_clase(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(10)), zero_division=0)
    return pd.DataFrame({"precision": p, "exhaustividad": r, "f1": f, "soporte": s}, index=CLASES)


reporte = reporte_por_clase(pred_test["y_true"], pred_test["y_pred"])
display(reporte.sort_values("f1").round(4))

In [ ]:
# ============================================================
# Reporte por clase en validación
# ============================================================
reporte_val = reporte_por_clase(pred_val["y_true"], pred_val["y_pred"])
display(reporte_val.sort_values("f1").round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Clases con menor F1 en test y en validación: ¿coinciden ambos conjuntos?
- Clases con precisión baja y exhaustividad alta (sobrepredicción) o al revés.
- Clases bien resueltas solo si precisión y exhaustividad son altas a la vez.

### 16.3 Matriz de confusión

Cada fila es la clase real y cada columna la predicha; la diagonal son los aciertos. Los valores fuera de la diagonal muestran qué pares de clases se confunden y en qué dirección.

In [ ]:
# ============================================================
# Matriz de confusión en test
# ============================================================
matriz = confusion_matrix(pred_test["y_true"], pred_test["y_pred"], labels=list(range(10)))
fig = px.imshow(matriz, x=CLASES, y=CLASES, text_auto=True, color_continuous_scale=[[0, "#FFFFFF"], [1, COLORES[0]]],
                labels={"x": "predicha", "y": "real", "color": "imágenes"}, height=600)
fig.update_layout(title="Matriz de confusión del modelo final (test)")
fig.show()

In [ ]:
# ============================================================
# Pares de clases más confundidos
# ============================================================
pares = [(CLASES[i], CLASES[j], matriz[i, j]) for i in range(10) for j in range(10) if i != j]
tabla_pares = pd.DataFrame(pares, columns=["real", "predicha", "imágenes"]).sort_values("imágenes", ascending=False)
display(tabla_pares.head(8).reset_index(drop=True))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Los pares más confundidos con sus conteos, y si la confusión es simétrica.
- Relación con las clases visualmente parecidas de la sección 3.4.

### 16.4 Análisis de errores

Se muestran los 10 errores de test con mayor confianza: imágenes en que el modelo asignó una probabilidad alta a una clase incorrecta. Permiten distinguir errores del modelo de imágenes ambiguas o mal etiquetadas.

In [ ]:
# ============================================================
# Errores con mayor confianza en test
# ============================================================
confianza = pred_test["probas"].max(1)
errores = np.where(pred_test["y_pred"] != pred_test["y_true"])[0]
peores = errores[np.argsort(-confianza[errores])][:10]
titulos = [f"{CLASES[pred_test['y_true'][i]]} → {CLASES[pred_test['y_pred'][i]]} ({confianza[i]:.2f})" for i in peores]
fig = px.imshow(x_test[peores], facet_col=0, facet_col_wrap=5, color_continuous_scale="gray_r",
                facet_row_spacing=0.1, height=420)
fig.for_each_annotation(lambda a: a.update(text=titulos[int(a.text.split("=")[1])], font_size=11))
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(title="Errores con mayor confianza (real → predicha, confianza)", coloraxis_showscale=False)
print(f"Errores en test: {len(errores)} de {len(y_test)} | con confianza ≥ 0,9: {(confianza[errores] >= 0.9).sum()}")
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Cuántos errores tienen confianza alta.
- Si las imágenes con error son ambiguas a simple vista o el modelo se equivoca en casos claros.

## 17. Comparación entre la configuración base y la final

Se muestran los hiperparámetros que cambiaron entre la configuración base y la final. El aporte de cada decisión está en la tabla de decisiones (sección 15.1). La comparación del desempeño con el mismo número de semillas se hace en la sección 19.3, porque el modelo final de una sola semilla, elegido entre muchas variantes, es una estimación optimista.

In [ ]:
# ============================================================
# Configuración base frente a final
# ============================================================
config_base_vs_final = pd.DataFrame({"base": pd.Series(CONFIG_BASE).astype(str), "final": pd.Series(CONFIG_FINAL).astype(str)})
config_base_vs_final["cambia"] = config_base_vs_final["base"] != config_base_vs_final["final"]
display(config_base_vs_final)

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Hiperparámetros que cambiaron y cuáles de esas decisiones superan MARGEN_ERROR.
- Decisiones tomadas en empate técnico y su criterio.

## 18. Síntesis de los experimentos

Tabla con todas las variantes entrenadas o recuperadas, en el orden de los experimentos.

In [ ]:
# ============================================================
# Resumen de todas las variantes
# ============================================================
resumen_general = pd.concat(resultados_experimentos, ignore_index=True)
display(resumen_general[["experimento"] + COLUMNAS_TABLA].round(4))

## 19. Verificaciones de la dependencia del orden

Elegir un hiperparámetro a la vez hace que cada decisión dependa de las anteriores. Sin usar test, se verifican dos combinaciones con interacción conocida sobre la configuración final y se repite el modelo final con las mismas 5 semillas que el base:

- **19.1 Tasa × lote:** al duplicar el lote, la tasa óptima suele subir, porque el gradiente promedio es menos ruidoso. Se prueban ambos duplicados a la vez.
- **19.2 Capacidad × regularización:** una red más grande puede necesitar más regularización. Se prueban el doble de filtros con 0,25 más de Dropout.
- **19.3 Configuración final con 5 semillas:** comparación justa con el base.

Una diferencia mayor que `MARGEN_ERROR` indica que el orden de las decisiones sí condicionó el resultado. Los experimentos ya hechos no se rehacen: se reinterpretan con estos resultados.

In [ ]:
# ============================================================
# Verificaciones 19.1 y 19.2
# ============================================================
VERIFICACIONES = {
    "19.1 Tasa ×2 con lote ×2": dict(learning_rate=CONFIG_FINAL["learning_rate"] * 2, batch_size=CONFIG_FINAL["batch_size"] * 2),
    "19.2 Filtros ×2 con Dropout +0,25": dict(filtros=CONFIG_FINAL["filtros"] * 2, dropout=min(0.75, CONFIG_FINAL["dropout"] + 0.25)),
}
filas_verificacion = [{"verificacion": "Configuración final", **fila_final}]
for nombre, cambios in VERIFICACIONES.items():
    fila, _, _ = entrenar_y_registrar({**CONFIG_FINAL, **cambios})
    print(f"{nombre}: {fila['origen']}")
    filas_verificacion.append({"verificacion": nombre, **fila})
tabla_verificacion = pd.DataFrame(filas_verificacion)
tabla_verificacion["diferencia_vs_final"] = tabla_verificacion["f1_macro_val"] - fila_final["f1_macro_val"]
tabla_verificacion["supera_margen"] = tabla_verificacion["diferencia_vs_final"].abs() > MARGEN_ERROR

In [ ]:
# ============================================================
# Resultados de las verificaciones 19.1 y 19.2
# ============================================================
display(tabla_verificacion[["verificacion", "f1_macro_val", "exactitud_val", "loss_val", "brecha_train_val",
                            "mejor_epoca", "parametros", "segundos_por_epoca", "diferencia_vs_final",
                            "supera_margen", "origen"]].round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- ¿Alguna combinación supera a la configuración final por más que MARGEN_ERROR?
- Si es así, qué decisión quedó condicionada por el orden (se decide con el usuario si se adopta).

### 19.3 Configuración final con 5 semillas

Se reentrena la configuración final con las mismas semillas adicionales del rango de error y se compara su distribución de F1 macro con la del modelo base.

In [ ]:
# ============================================================
# Reentrenamientos de la configuración final
# ============================================================
filas_final_semillas = [{"semilla": SEED, **fila_final}]
for semilla in SEMILLAS_ADICIONALES:
    fila, _, _ = entrenar_y_registrar(CONFIG_FINAL, semilla=semilla)
    print(f"Semilla {semilla}: {fila['origen']}")
    filas_final_semillas.append({"semilla": semilla, **fila})
tabla_final_semillas = pd.DataFrame(filas_final_semillas)

In [ ]:
# ============================================================
# Comparación justa: base frente a final con 5 semillas
# ============================================================
comparacion_justa = pd.DataFrame({
    "base": tabla_variabilidad["f1_macro_val"].agg(["mean", "std", "min", "max"]),
    "final": tabla_final_semillas["f1_macro_val"].agg(["mean", "std", "min", "max"]),
})
comparacion_justa["diferencia"] = comparacion_justa["final"] - comparacion_justa["base"]
print(f"Mejora media: {comparacion_justa.loc['mean', 'diferencia'] * 100:.2f} pp | MARGEN_ERROR: {MARGEN_ERROR * 100:.2f} pp")
display(comparacion_justa.round(4))

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Mejora media de F1 macro con 5 semillas frente a la estimación de una sola semilla (cota optimista).
- ¿Los rangos de base y final se superponen?
- Advertencia: muchas variantes se evaluaron sobre la misma validación, así que su F1 final es optimista; el test es la estimación independiente.

## 20. Representación visual para la fusión

El notebook de fusión no usa las predicciones de la CNN, sino su **representación**: el vector de la capa densa, con `unidades_densas` valores por imagen. Para comprobar que ese vector separa las clases, se proyectan las representaciones de validación a dos dimensiones con **PCA** (análisis de componentes principales) y se grafican coloreadas por clase.

`extraer_representaciones(modelo, x, config)` aplica `representacion()` por lotes, sin gradientes. Retorna un arreglo de NumPy de forma `[imágenes, unidades_densas]`.

In [ ]:
# ============================================================
# Extracción de representaciones y proyección PCA
# ============================================================
def extraer_representaciones(modelo, x, config, lote=2048):
    modelo.eval()
    salidas = []
    with torch.no_grad():
        for inicio in range(0, len(x), lote):
            salidas.append(modelo.representacion(preparar_lote(x[inicio:inicio + lote], config)).cpu())
    return torch.cat(salidas).numpy()


representaciones_val = extraer_representaciones(modelo_final, X_VAL, CONFIG_FINAL)
pca = PCA(n_components=2, random_state=SEED)
proyeccion = pca.fit_transform(representaciones_val)
print(f"Forma de la representación: {representaciones_val.shape} | varianza explicada por 2 componentes: "
      f"{pca.explained_variance_ratio_.sum() * 100:.1f} %")

In [ ]:
# ============================================================
# Proyección de la representación por clase
# ============================================================
fig = px.scatter(x=proyeccion[:, 0], y=proyeccion[:, 1], color=[CLASES[c] for c in y_val], opacity=0.6,
                 labels={"x": "componente 1", "y": "componente 2", "color": "clase"}, height=560)
fig.update_traces(marker_size=4)
fig.update_layout(title="Representación de la CNN en validación (proyección PCA)")
fig.show()

**Observación / Hallazgo:** *(completar después de ejecutar)*

- Clases que forman grupos separados y clases que se superponen (comparar con la matriz de confusión).
- Varianza explicada por dos componentes: cuánto de la representación se ve en el gráfico.

## 21. Conclusiones generales

*(completar después de ejecutar, con las cifras obtenidas: desempeño en test frente a las líneas base, aporte de la estructura convolucional, efecto del preprocesamiento, de las funciones y de los hiperparámetros de entrenamiento, decisiones en empate y limitaciones)*

### 21.1 Mejoras propuestas

*(completar después de ejecutar)*

## 22. Conclusión final

*(completar después de ejecutar)*

## 23. Referencias

- Hendrycks, D. y Gimpel, K. (2016). *Gaussian Error Linear Units (GELUs)*. arXiv:1606.08415.
- Ioffe, S. y Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. ICML. arXiv:1502.03167.
- Kingma, D. P. y Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR. arXiv:1412.6980.
- LeCun, Y., Bottou, L., Bengio, Y. y Haffner, P. (1998). *Gradient-Based Learning Applied to Document Recognition*. Proceedings of the IEEE, 86(11), 2278–2324.
- Simonyan, K. y Zisserman, A. (2015). *Very Deep Convolutional Networks for Large-Scale Image Recognition*. ICLR. arXiv:1409.1556.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. y Salakhutdinov, R. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR, 15, 1929–1958.
- Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J. y Wojna, Z. (2016). *Rethinking the Inception Architecture for Computer Vision*. CVPR. arXiv:1512.00567.
- Xiao, H., Rasul, K. y Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*. arXiv:1708.07747.
- PyTorch. *Documentación oficial*. https://pytorch.org/docs/stable/
- *EFT125_DLY0100_Estudiante* y *ET_Ejemplo_sentimiento_junto_clasificacion*. Material proporcionado por Duoc UC, 2026, en Deep Learning (DLY0100).

## 24. Anexo A — Síntesis visual para la presentación

Gráficos en formato 16:9 con la paleta de la presentación, generados desde las variables ya calculadas (no entrenan). Cada uno se exporta a HTML en `respaldo_eft_cnn/graficos/`; el PNG se obtiene con el botón de cámara del gráfico o directamente si está instalado `kaleido`.

`estilo_presentacion(fig, titulo, subtitulo)` aplica el estilo común y retorna la figura. `exportar(fig, nombre)` guarda el gráfico en HTML (y en PNG si es posible).

In [ ]:
# ============================================================
# Estilo y exportación de los gráficos de la presentación
# ============================================================
RUTA_RESPALDO = "respaldo_eft_cnn"
RUTA_GRAFICOS = f"{RUTA_RESPALDO}/graficos"
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
AZUL_MARINO = "#0E2747"
ANCHO_PNG, ALTO_PNG = 1600, 900


def estilo_presentacion(fig, titulo, subtitulo=None):
    texto = f"<b>{titulo}</b>"
    if subtitulo:
        texto += f"<br><span style='font-size:15px;color:#555555'>{subtitulo}</span>"
    fig.update_layout(template="plotly_white", title={"text": texto, "x": 0.02},
                      font={"family": "Arial", "size": 16, "color": AZUL_MARINO},
                      height=560, margin={"t": 110, "l": 80, "r": 40, "b": 80},
                      legend={"orientation": "h", "y": -0.18, "title": None})
    return fig


def exportar(fig, nombre):
    fig.write_html(f"{RUTA_GRAFICOS}/{nombre}.html", include_plotlyjs="cdn",
                   config={"toImageButtonOptions": {"format": "png", "width": ANCHO_PNG, "height": ALTO_PNG,
                                                    "scale": 2, "filename": nombre}})
    try:
        fig.write_image(f"{RUTA_GRAFICOS}/{nombre}.png", width=ANCHO_PNG, height=ALTO_PNG, scale=2)
        print(f"{nombre}: guardado en HTML y PNG")
    except Exception:
        print(f"{nombre}: guardado en HTML (PNG desde el botón de cámara del gráfico)")

### A.1 CNN frente a MLP

In [ ]:
# ============================================================
# A.1 CNN frente a MLP
# ============================================================
datos = tabla_de("arquitectura")
fig = go.Figure(go.Bar(x=datos["variante"], y=datos["f1_macro_val"], marker_color=COLORES[0],
                       text=[f"{v:.3f}<br>{p / 1000:.0f} mil parámetros" for v, p in zip(datos["f1_macro_val"], datos["parametros"])],
                       textposition="outside"))
fig.update_yaxes(title_text="F1 macro (validación)", range=[datos["f1_macro_val"].min() - 0.03, 1])
estilo_presentacion(fig, "La estructura convolucional supera al MLP", "F1 macro de validación con número de parámetros comparable")
exportar(fig, "a1_cnn_vs_mlp")
fig.show()

### A.2 Funciones de activación

In [ ]:
# ============================================================
# A.2 Funciones de activación: F1 de validación por época
# ============================================================
fig = go.Figure()
paleta = COLORES + [AZUL_MARINO, AMARILLO]
for i, (etiqueta, historia) in enumerate(historias_activacion.items()):
    fig.add_trace(go.Scatter(x=list(range(1, len(historia["f1_val"]) + 1)), y=historia["f1_val"], name=etiqueta,
                             line={"color": paleta[i % len(paleta)], "width": 3}))
fig.update_xaxes(title_text="época")
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Efecto de la función de activación", "F1 macro de validación por época")
exportar(fig, "a2_activaciones")
fig.show()

### A.3 Funciones de pérdida

In [ ]:
# ============================================================
# A.3 Funciones de pérdida: F1 de validación por época
# ============================================================
fig = go.Figure()
for i, (etiqueta, historia) in enumerate(historias_perdida.items()):
    fig.add_trace(go.Scatter(x=list(range(1, len(historia["f1_val"]) + 1)), y=historia["f1_val"], name=etiqueta,
                             line={"color": (COLORES + [AMARILLO])[i % 4], "width": 3}))
fig.update_xaxes(title_text="época")
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Efecto de la función de pérdida", "F1 macro de validación por época")
exportar(fig, "a3_perdidas")
fig.show()

### A.4 Tasa de aprendizaje

In [ ]:
# ============================================================
# A.4 Tasa de aprendizaje
# ============================================================
datos = tabla_de("tasa de aprendizaje").assign(tasa=lambda d: d["variante"].astype(float)).sort_values("tasa")
fig = go.Figure(go.Scatter(x=datos["variante"], y=datos["f1_macro_val"], mode="lines+markers+text",
                           text=[f"{v:.3f}" for v in datos["f1_macro_val"]], textposition="top center",
                           line={"color": COLORES[0], "width": 3}, marker_size=12))
fig.update_xaxes(title_text="tasa de aprendizaje", type="category")
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Tasa de aprendizaje", "F1 macro de validación según la tasa")
exportar(fig, "a4_tasa")
fig.show()

### A.5 Tamaño del lote: desempeño y costo

In [ ]:
# ============================================================
# A.5 Tamaño del lote: F1 frente a segundos por época
# ============================================================
datos = tabla_de("tamaño del lote").assign(lote=lambda d: d["variante"].astype(int)).sort_values("lote")
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=datos["variante"], y=datos["segundos_por_epoca"], name="segundos por época",
                     marker_color=COLORES[2]), secondary_y=True)
fig.add_trace(go.Scatter(x=datos["variante"], y=datos["f1_macro_val"], name="F1 macro (validación)", mode="lines+markers",
                         line={"color": COLORES[0], "width": 3}, marker_size=12), secondary_y=False)
fig.update_xaxes(title_text="tamaño del lote", type="category")
fig.update_yaxes(title_text="F1 macro (validación)", secondary_y=False)
fig.update_yaxes(title_text="segundos por época", secondary_y=True)
estilo_presentacion(fig, "Tamaño del lote: desempeño frente a costo")
exportar(fig, "a5_lote")
fig.show()

### A.6 Épocas sin Early Stopping

In [ ]:
# ============================================================
# A.6 Épocas sin Early Stopping
# ============================================================
epocas = list(range(1, len(historia_epocas["loss_val"]) + 1))
mejor = int(np.argmin(historia_epocas["loss_val"])) + 1
fig = go.Figure([go.Scatter(x=epocas, y=historia_epocas["loss_train"], name="entrenamiento", line={"color": COLORES[2], "width": 3}),
                 go.Scatter(x=epocas, y=historia_epocas["loss_val"], name="validación", line={"color": COLORES[0], "width": 3})])
fig.add_vline(x=mejor, line_dash="dash", line_color=AMARILLO, annotation_text=f"mejor época: {mejor}")
fig.update_xaxes(title_text="época")
fig.update_yaxes(title_text="pérdida (entropía cruzada)")
estilo_presentacion(fig, "Dónde empieza el sobreajuste", "Pérdida por época sin Early Stopping")
exportar(fig, "a6_epocas")
fig.show()

### A.7 Modelo base frente a modelo final (5 semillas)

In [ ]:
# ============================================================
# A.7 Base frente a final con 5 semillas
# ============================================================
fig = go.Figure()
for nombre, tabla, color in [("Base", tabla_variabilidad, COLORES[2]), ("Final", tabla_final_semillas, COLORES[0])]:
    fig.add_trace(go.Box(y=tabla["f1_macro_val"], name=nombre, marker_color=color, boxpoints="all", jitter=0.3))
fig.update_yaxes(title_text="F1 macro (validación)")
estilo_presentacion(fig, "Configuración base frente a final", "F1 macro de validación con las mismas 5 semillas")
exportar(fig, "a7_base_vs_final")
fig.show()

### A.8 Impacto de cada decisión

In [ ]:
# ============================================================
# A.8 Aporte de cada decisión
# ============================================================
datos = tabla_decisiones.dropna(subset=["aporte"])
fig = go.Figure(go.Bar(x=datos["experimento"], y=datos["aporte"] * 100,
                       marker_color=[COLORES[0] if s else COLORES[2] for s in datos["supera_margen"]],
                       text=[f"{v * 100:+.2f}" for v in datos["aporte"]], textposition="outside"))
fig.add_hline(y=MARGEN_ERROR * 100, line_dash="dash", line_color=AMARILLO, annotation_text="rango de error")
fig.update_yaxes(title_text="aporte al F1 macro (pp)")
estilo_presentacion(fig, "Aporte de cada decisión", "Azul oscuro: supera el rango de error")
exportar(fig, "a8_decisiones")
fig.show()

### A.9 Matriz de confusión en test

In [ ]:
# ============================================================
# A.9 Matriz de confusión normalizada por clase real
# ============================================================
matriz_pct = matriz / matriz.sum(1, keepdims=True) * 100
fig = px.imshow(matriz_pct, x=CLASES, y=CLASES, text_auto=".0f", color_continuous_scale=[[0, "#FFFFFF"], [1, COLORES[0]]],
                labels={"x": "predicha", "y": "real", "color": "%"})
estilo_presentacion(fig, "Matriz de confusión en test", "Porcentaje de cada clase real")
fig.update_layout(height=700)
exportar(fig, "a9_matriz_confusion")
fig.show()

## 25. Anexo B — Herramientas y tecnologías de apoyo utilizadas

| Categoría | Herramienta | Uso |
|---|---|---|
| Entorno de ejecución | Google Colab con GPU NVIDIA T4 | Entrenamiento de todos los modelos |
| Editor | Visual Studio Code con la extensión de Google Colab | Edición y ejecución del notebook conectado al servidor de Colab |
| Asistente de IA | Claude Opus 5.5 (Anthropic) | Estructuración del notebook, apoyo en el código y en la redacción; los resultados, decisiones e interpretaciones se revisaron con las salidas del propio notebook |
| Framework | PyTorch (https://pytorch.org) | Definición y entrenamiento de la CNN y del MLP |
| Librerías | NumPy (https://numpy.org), Pandas (https://pandas.pydata.org), scikit-learn (https://scikit-learn.org), Plotly (https://plotly.com/python/) | Arreglos, tablas, métricas, PCA y gráficos |
| Repositorio | GitHub | Datos, resultados guardados y versiones del notebook |

## 26. Respaldo de resultados

Se guardan en `respaldo_eft_cnn/` y se comprimen en `respaldo_eft_cnn.zip` todas las tablas, las historias, los índices de la partición y el modelo final. El archivo `cnn_final.pt` es el que usa el notebook de fusión: contiene la configuración, los pesos, los estadísticos de normalización y los índices de entrenamiento y validación.

Con un servidor remoto (Colab), los archivos quedan en el servidor: hay que descargar el `.zip` y `resultados_cnn_guardados.json` desde el panel de archivos antes de cerrar la sesión.

In [ ]:
# ============================================================
# Respaldo de resultados
# ============================================================
os.makedirs(RUTA_RESPALDO, exist_ok=True)
resumen_general.to_csv(f"{RUTA_RESPALDO}/experimentos_todas_las_ejecuciones.csv", index=False)
tabla_variabilidad.to_csv(f"{RUTA_RESPALDO}/variabilidad.csv", index=False)
estadisticos_variabilidad.to_csv(f"{RUTA_RESPALDO}/estadisticos_variabilidad.csv")
tabla_lineas_base_val.to_csv(f"{RUTA_RESPALDO}/lineas_base_validacion.csv", index=False)
tabla_epocas.to_csv(f"{RUTA_RESPALDO}/epocas_sin_early_stopping.csv")
tabla_decisiones.to_csv(f"{RUTA_RESPALDO}/decisiones.csv", index=False)
config_base_vs_final.to_csv(f"{RUTA_RESPALDO}/config_base_vs_final.csv")
tabla_verificacion.to_csv(f"{RUTA_RESPALDO}/verificaciones_orden.csv", index=False)
tabla_final_semillas.to_csv(f"{RUTA_RESPALDO}/final_5_semillas.csv", index=False)
comparacion_justa.to_csv(f"{RUTA_RESPALDO}/comparacion_base_final.csv")
tabla_test.to_csv(f"{RUTA_RESPALDO}/metricas_test.csv", index=False)
reporte.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase_test.csv")
reporte_val.to_csv(f"{RUTA_RESPALDO}/metricas_por_clase_validacion.csv")
pd.DataFrame(matriz, index=CLASES, columns=CLASES).to_csv(f"{RUTA_RESPALDO}/matriz_confusion.csv")
with open(f"{RUTA_RESPALDO}/historias.json", "w") as f:
    json.dump({"base": historia_base, "final": historia_final, "sin_early_stopping": historia_epocas,
               "activacion": historias_activacion, "perdida": historias_perdida}, f)
np.save(f"{RUTA_RESPALDO}/y_test_proba.npy", pred_test["probas"])
torch.save({"config": CONFIG_FINAL, "estado": modelo_final.state_dict(), "media_pixel": MEDIA_PIXEL,
            "std_pixel": STD_PIXEL, "clases": CLASES, "idx_train": idx_train, "idx_val": idx_val},
           f"{RUTA_RESPALDO}/cnn_final.pt")
guardar_corridas()
shutil.copy(ARCHIVO_RESULTADOS, f"{RUTA_RESPALDO}/{ARCHIVO_RESULTADOS}")
shutil.make_archive(RUTA_RESPALDO, "zip", RUTA_RESPALDO)
print(f"Respaldo guardado en {RUTA_RESPALDO}.zip ({os.path.getsize(RUTA_RESPALDO + '.zip') / 1e6:.1f} MB)")

## 27. Descripción de las salidas

| Archivo | Contenido | Uso posterior |
|---|---|---|
| `resultados_cnn_guardados.json` | Métricas de cada corrida, por clave de configuración y semilla | Subirlo a la raíz del repositorio para que la siguiente ejecución no repita corridas |
| `respaldo_eft_cnn/cnn_final.pt` | Configuración, pesos, estadísticos de normalización, clases e índices de la partición del modelo final | Lo carga el notebook de fusión para extraer la representación visual |
| `respaldo_eft_cnn/*.csv` | Tablas de experimentos, variabilidad, decisiones, verificaciones, test y matriz de confusión | Regenerar tablas y gráficos sin reentrenar |
| `respaldo_eft_cnn/historias.json` | Curvas por época del base, del final, sin Early Stopping y de las activaciones y pérdidas | Gráficos de la presentación |
| `respaldo_eft_cnn/y_test_proba.npy` | Probabilidades del modelo final en test | Análisis de errores y calibración |
| `respaldo_eft_cnn/graficos/` | Gráficos del anexo A en HTML (y PNG si hay `kaleido`) | Diapositivas de la presentación |
| `respaldo_eft_cnn.zip` | Todo lo anterior comprimido | Descarga desde Colab |